# ⚠️ Notice

## This material is the intellectual property of <font color="cyan">Dr. Lysa V. Comia.</font>

Redistribution, reproduction, or use beyond personal reference is strictly prohibited without the prior written consent of the author.

The two-fruit training cells below were prepared separately for this group's project. Keep Dr. Comia's notice and credit with any personal working copy based on her trainer.

# Two-Stage Instance Segmentation of Post-Harvest Fruit Defects with YOLO26

**Group:** CIPHER | Artificial Intelligence 2 | AM3 | Mapúa University

**Members:** Ralph Kevin G. Morales, Alexander Jann P. Espia, Deangelo James P. Largueza, Niel Francis M. Arligue

**Adviser:** Dr. Lysa V. Comia

The training cells are adapted from Dr. Comia's YOLOv26 trainer. Her notices are kept at the top and bottom.

## Introduction

**Task.** Segment every clearly visible surface defect on apple and tomato photos at the pixel level and name its type.
The output is one mask per defect region. This is instance segmentation.

**Why it matters.** Post-harvest grading datasets label a whole photo with a grade. A grade does not say where the
damage is or how large it is. A defect mask gives both, which supports sorting and quality checks.

**Approach.** Two stages adapted from Leiva et al. (2026):

1. Stage 1 trains YOLO26l-seg to segment the whole fruit on every training image.
2. Stage 2 starts from the Stage 1 best checkpoint and trains three defect classes.

The classes describe visible appearance only. They are not a disease diagnosis.

![Pipeline](../reports/figures/pipeline.png)

## Dataset

| Item | Apple |
|---|---|
| Sources | AFruitDB grading photos (986 train files), Lab2Wild apple rot set (843), team photos (281) |
| Original labels | AFruitDB has image-level grades only. They were not used as targets |
| Pixel masks | Drawn by the team in Roboflow and a SAM-assisted tool. Every mask reviewed by a member |
| Classes | `bruise_discoloration`, `rot_mold_decay`, `surface_damage` (plus `apple` for Stage 1) |
| Format | YOLO segmentation polygons, one `.txt` per image |
| Train | 2,110 files from 726 capture groups |
| Validation | 215 images from 210 capture groups |
| Test | 79 images from 75 capture groups |
| Reserve | 63 images, never used |

| Class | Train images | Train masks |
|---|---:|---:|
| bruise_discoloration | 607 | 709 |
| rot_mold_decay | 620 | 697 |
| surface_damage | 547 | 873 |

**Preprocessing.** Roboflow auto-orient and resize. The training export contains up to three augmented variants of
many photos so 2,110 files are not 2,110 distinct photos. A capture group holds all files from one original photo and
never crosses splits. Exact pixel hashes were checked across splits. 545 training images have no defect and act as
negatives.

**Augmentation.** Online during training only: flips (p 0.5), rotation up to 10 degrees, translate 0.05, scale 0.1,
HSV 0.01 0.2 0.2. Mosaic, MixUp and Copy-Paste are off.

| Item | Tomato | Apple + tomato (combined) |
|---|---|---|
| Sources | Team Roboflow tomato project (export declares CC BY 4.0) | Both sets merged without reshuffling |
| Train | 1,313 files from 459 capture groups | 3,423 files from 1,185 capture groups |
| Validation | 132 images | 347 images |
| Test | 95 images | 174 images (79 apple, 95 tomato) |
| Reserve | 39 images | 102 images |

Tomato and the combined set use the same three defect classes, the same preprocessing and the same two-stage recipe.

![Defect gallery](../reports/figures/defect_gallery.png)

Team-drawn masks on training images. Green outline is the whole fruit (Stage 1). Filled regions are defects (Stage 2).

![Class distribution](../reports/figures/class_distribution.png)

Every defect class has more than 570 training masks on each fruit. Defect size differs a lot: apple rot covers a
median 48% of the fruit, tomato rot 3.5% and surface damage under 0.5% on both fruits. Small masks are harder to
segment.

## Requirements

| Library | Version | Use |
|---|---|---|
| ultralytics | 8.4.126 | YOLO26 training, validation and prediction |
| torch | Colab default (2.x, CUDA) | Deep learning backend |
| numpy, pandas | Colab default | Arrays and result tables |
| pyyaml | Colab default | Dataset and run config files |
| pillow, opencv | Colab default | Image loading and mask drawing |
| streamlit | 1.50 or newer | Web app (local) |

Hardware: Google Colab with one NVIDIA A100 80 GB GPU. Data and runs are stored in `MyDrive/YOLOv26/`.

## Run configuration

All settings live in the next cell. Each fruit has a Stage 1 and a Stage 2 Run ID:

| Model | Stage 1 | Stage 2 |
|---|---|---|
| Apple | Run 21 | Run 22 |
| Tomato | Run 23 | Run 24 |
| Apple + tomato (one model) | Run 25 | Run 26 |

`PILOT_HISTORY` and `RUN_HISTORY` in the next cell record the dataset, model and settings of every run from the
first pilot to Run 26. They are a record only. To train a stage change only `RUN_ID`. `RUN_FINAL_TEST` stays `False` until the checkpoint is frozen. The saved
outputs below are from the last session, which loaded Run 26 and ran its final test.

In [17]:
from pathlib import Path
import os

# Runs 23/24: tomato. Runs 25/26: apple and tomato together, then three shared defects.
RUN_ID = 26
SAVED_RUN = ''
RUN_FINAL_TEST = True
# Every training run from the first pilot to the final models. Pilots P1 to P6 had no Run ID.
# Runs 1 to 20 used earlier versions of this notebook and earlier datasets. They are kept as a
# record only. RUN_PLANS below holds the runs this notebook can train or resume.
PILOT_HISTORY = {
    'P1': dict(dataset='Fruit-Segmentation-v5', fruits='apple', model='yolo26n-seg', imgsz=1024,
             epochs='50/50', batch=8, optimizer='auto', lr0=0.01, result='val mask mAP50 0.474'),
    'P2': dict(dataset='Fruit-Segmentation-v5 combined', fruits='apple', model='yolo26n-seg',
             imgsz=1024, epochs='96/100', batch=8, optimizer='auto', lr0=0.01,
             result='val mask mAP50 0.751'),
    'P3': dict(dataset='apple-pilot-export5', fruits='apple', model='yolo26n-seg', imgsz=1024,
             epochs='50/50', batch=8, optimizer='auto', lr0=0.01, result='val mask mAP50 0.498'),
    'P4': dict(dataset='apple-mask-and-defect-pilot', fruits='apple', model='yolo26n-seg', imgsz=1024,
             epochs='50/50', batch=8, optimizer='auto', lr0=0.01, result='val mask mAP50 0.700'),
    'P5': dict(dataset='three-fruit-first-pilot', fruits='apple+mango+tomato', model='yolo26s-seg',
             imgsz=1024, epochs='50/50', batch=4, optimizer='auto', lr0=0.01,
             result='val mask mAP50 0.683'),
    'P6': dict(dataset='apple-tomato-online-aug-v1', fruits='apple+tomato', model='yolo26s-seg',
             imgsz=1024, epochs='100/100', batch='auto', optimizer='auto', lr0=0.01,
             result='val mask mAP50 0.687'),
}
RUN_HISTORY = {
    1: dict(dataset='apple-tomato-sep26', fruits='apple+tomato', model='yolo26s-seg', imgsz=1024,
             epochs='50/50', batch=16, optimizer='SGD', lr0=0.01, result='val mask mAP50 0.626'),
    2: dict(dataset='apple-tomato-sep26', fruits='apple+tomato', model='yolo26s-seg', imgsz=1024,
             epochs='50/50', batch=16, optimizer='SGD', lr0=0.003, result='val mask mAP50 0.739'),
    '2b': dict(dataset='apple-tomato-sep26', fruits='apple+tomato', model='yolo26s-seg', imgsz=1024,
             epochs='100/100', batch=16, optimizer='AdamW', lr0=0.001, result='val mask mAP50 0.742'),
    3: dict(dataset='apple-tomato-sep26', fruits='apple+tomato', model='yolo26s-seg', imgsz=1024,
             epochs='50/50', batch=16, optimizer='AdamW', lr0=0.001, extra_conv=True,
             result='val mask mAP50 0.611'),
    4: dict(result='Run ID not used. No run folder on Drive'),
    5: dict(dataset='apple-tomato-sep27-members', fruits='apple+tomato', model='yolo26s-seg',
             imgsz=1024, epochs='100/100', batch=16, optimizer='SGD', lr0=0.01,
             result='val mask mAP50 0.747'),
    6: dict(dataset='apple-tomato-sep27-members 70/20/10', fruits='apple+tomato', model='yolo26m-seg',
             imgsz=1024, epochs='100/100', batch=8, optimizer='SGD', lr0=0.01,
             result='val mask mAP50 0.750'),
    7: dict(dataset='apple-tomato-sep27-members 70/20/10', fruits='apple+tomato', model='yolo26l-seg',
             imgsz=1024, epochs='35/100', batch=2, optimizer='SGD', lr0=0.01,
             result='val mask mAP50 0.647'),
    8: dict(result='Run ID not used. No run folder on Drive'),
    9: dict(dataset='apple-tomato-sep29-refresh', fruits='apple+tomato', model='yolo26m-seg', imgsz=1024,
            epochs='2/100', batch=3, optimizer='SGD', lr0=0.01,
            result='stopped after 2 epochs on an L4 GPU. Replaced by Run 11'),
    10: dict(dataset='apple-tomato-sep29-refresh, then apple-sep30-fruit9', model='yolo26m-seg', imgsz=1024,
             epochs='0/100', batch=16, optimizer='SGD', lr0=0.01, patience=0,
             result='two attempts stopped before the first epoch finished'),
    11: dict(dataset='apple-tomato-sep29-refresh', fruits='apple+tomato', model='yolo26m-seg',
             imgsz=1024, epochs='100/100', batch=16, optimizer='SGD', lr0=0.01,
             result='val mask mAP50 0.715'),
    12: dict(dataset='apple-sep30-fruit9', fruits='apple', model='yolo26m-seg', imgsz=1024, epochs='0/100',
             batch=16, optimizer='SGD', lr0=0.01, patience=0, result='stopped before the first epoch finished'),
    13: dict(dataset='apple-sep30-fruit9', fruits='apple', model='yolo26m-seg', imgsz=768, epochs='0/100',
             batch=8, optimizer='SGD', lr0=0.01, patience=0, result='stopped before the first epoch finished'),
    14: dict(dataset='apple-sep30-fruit9', fruits='apple', model='yolo26m-seg', imgsz=1024,
             epochs='100/100', batch=16, optimizer='SGD', lr0=0.01, result='val mask mAP50 0.714'),
    15: dict(dataset='apple-one-defect-oct01', fruits='apple', model='yolo26m-seg', imgsz=1024,
             epochs='81/100', batch=16, optimizer='SGD', lr0=0.01, result='val mask mAP50 0.673'),
    16: dict(result='Run ID not used. No run folder on Drive'),
    17: dict(stage=1, optimizer='SGD', lr0=0.001, result='planned in RUN_PLANS, never trained'),
    18: dict(stage=2, optimizer='AdamW', lr0=0.0005, result='planned in RUN_PLANS, never trained'),
    19: dict(dataset='apple-stage1-whole-oct01', fruits='apple', model='yolo26m-seg', imgsz=864,
             epochs='21/100', batch=56, optimizer='SGD', lr0=0.001, result='val mask mAP50 0.985'),
    20: dict(dataset='apple-stage2-defects-oct01', fruits='apple', model='Run 19 best.pt (26m)',
             imgsz=864, epochs='91/300', batch=56, optimizer='AdamW', lr0=0.0005,
             result='val mask mAP50 0.671'),
    21: dict(dataset='apple-stage1-whole-oct01', fruits='apple', model='yolo26l-seg', imgsz=864,
             epochs='34/100', batch=56, optimizer='SGD', lr0=0.001, result='val mask mAP50 0.985'),
    22: dict(dataset='apple-stage2-defects-oct01', fruits='apple', model='Run 21 best.pt (26l)',
             imgsz=864, epochs='124/300', batch=56, optimizer='AdamW', lr0=0.0005,
             result='val mask mAP50 0.688, test 0.613. Final apple model'),
    23: dict(dataset='tomato-stage1-whole-oct01', fruits='tomato', model='yolo26l-seg', imgsz=864,
             epochs='14/100', batch=56, optimizer='SGD', lr0=0.001, result='val mask mAP50 0.966'),
    24: dict(dataset='tomato-stage2-defects-oct01', fruits='tomato', model='Run 23 best.pt (26l)',
             imgsz=864, epochs='190/300', batch=56, optimizer='AdamW', lr0=0.0005,
             result='val mask mAP50 0.494, test 0.393. Final tomato model'),
    25: dict(dataset='apple-tomato-stage1-whole-oct01', fruits='apple+tomato', model='yolo26l-seg',
             imgsz=864, epochs='14/100', batch=56, optimizer='SGD', lr0=0.001,
             result='val mask mAP50 apple 0.985 tomato 0.949'),
    26: dict(dataset='apple-tomato-stage2-defects-oct01', fruits='apple+tomato',
             model='Run 25 best.pt (26l)', imgsz=864, epochs='97/100', batch=56, optimizer='AdamW',
             lr0=0.0005, result='val mask mAP50 0.600, test apple 0.586 tomato 0.352. Combined model'),
}

RUN_PLANS = {
    17: dict(stage=1, optimizer='SGD', lr0=0.001, epochs=100,
             patience=10, momentum=0.9, source_run_id=None),
    18: dict(stage=2, optimizer='AdamW', lr0=0.0005, epochs=300,
             patience=30, momentum=0.9, source_run_id=17),
    19: dict(stage=1, optimizer='SGD', lr0=0.001, epochs=100,
             patience=10, momentum=0.9, source_run_id=None),
    20: dict(stage=2, optimizer='AdamW', lr0=0.0005, epochs=300,
             patience=30, momentum=0.9, source_run_id=19),
    21: dict(stage=1, optimizer='SGD', lr0=0.001, epochs=100,
             patience=10, momentum=0.9, source_run_id=None),
    22: dict(stage=2, optimizer='AdamW', lr0=0.0005, epochs=300,
             patience=30, momentum=0.9, source_run_id=21),
    23: dict(fruit='tomato', model='yolo26l-seg.pt', stage=1,
             optimizer='SGD', lr0=0.001, epochs=100,
             patience=10, momentum=0.9, source_run_id=None),
    24: dict(fruit='tomato', model='yolo26l-seg.pt', stage=2,
             optimizer='AdamW', lr0=0.0005, epochs=300,
             patience=30, momentum=0.9, source_run_id=23),
    25: dict(fruit='apple_tomato', model='yolo26l-seg.pt', stage=1,
             optimizer='SGD', lr0=0.001, epochs=100,
             patience=5, momentum=0.9, source_run_id=None),
    26: dict(fruit='apple_tomato', model='yolo26l-seg.pt', stage=2,
             optimizer='AdamW', lr0=0.0005, epochs=100,
             patience=30, momentum=0.9, source_run_id=25),
}
if RUN_ID not in RUN_PLANS:
    raise ValueError('Add the new RUN_ID and its settings to RUN_PLANS in Cell 2.')
PLAN = dict(RUN_PLANS[RUN_ID])
STAGE = int(PLAN['stage'])
if STAGE not in (1, 2):
    raise ValueError('Stage must be 1 (whole fruit) or 2 (three defects).')

# Historical plans without a fruit field continue to select apple.
FRUIT = PLAN.get('fruit', 'apple')
DATASET_PROFILES = {
    'apple': dict(
        source='apple-sep30-fruit9',
        sha256='779e8d8cb90aa25b4a83b2a6be99b0fc6ea6417426f3cc58488a4850ac0ac85e',
        stages={1: 'apple-stage1-whole-oct01', 2: 'apple-stage2-defects-oct01'},
    ),
    'tomato': dict(
        source='tomato-sep29-source-oct01',
        sha256='0f7093ee63dffe610ee837a332d75613375af217b513a43ffe317be0a3df9155',
        stages={1: 'tomato-stage1-whole-oct01', 2: 'tomato-stage2-defects-oct01'},
    ),
    'apple_tomato': dict(
        fruits=['apple', 'tomato'],
        source='apple-tomato-source-oct01',
        sha256='1bc76dc6eb6db1eedfe5bdf439cc8409b1eb6f8d62f76ea659a3753397e8e5d2',
        stages={1: 'apple-tomato-stage1-whole-oct01',
                2: 'apple-tomato-stage2-defects-oct01'},
    ),
}
if FRUIT not in DATASET_PROFILES:
    raise ValueError('Add the fruit and its source archive to DATASET_PROFILES in Cell 2.')
PROFILE = DATASET_PROFILES[FRUIT]
SOURCE_DATASET_NAME = PROFILE['source']
STAGE_DATASETS = dict(PROFILE['stages'])
WHOLE_FRUIT_CLASSES = list(PROFILE.get('fruits', [FRUIT]))
STAGE_CLASSES = {
    1: WHOLE_FRUIT_CLASSES,
    2: ['bruise_discoloration', 'rot_mold_decay', 'surface_damage'],
}
SOURCE_CLASSES = [*WHOLE_FRUIT_CLASSES, *STAGE_CLASSES[2]]
DATASET_NAME = STAGE_DATASETS[STAGE]
EXPECTED_CLASSES = STAGE_CLASSES[STAGE]
EXPECTED_SHA256 = PROFILE['sha256']
MODEL_NAME = PLAN.get('model', 'yolo26l-seg.pt')
ULTRALYTICS_VERSION = '8.4.126'
IMG_SIZE = 864
BATCH_SIZE = 56  # Explicit A100 80GB batch; based on measured Run 19 memory.
SEED = 0
WORKERS = min(16, os.cpu_count() or 2)  # Use available cores, up to 16 loader workers.
DEVICE = 0
AMP = True
DETERMINISTIC = False
REQUIRE_A100 = True
CACHE = 'ram'
TF32 = True
CUDNN_BENCHMARK = False  # Avoid algorithm-search stalls at new batch/validation shapes.
EVAL_BATCH_CAP = 48  # Also caps validation batches during training.
MIN_GPU_GIB = 70  # These batch settings target A100 80GB, not A100 40GB.
ALLOW_HARDWARE_RECONFIGURE = True  # Continue same-task runs with logged hardware changes.
HARDWARE_CONFIG_KEYS = {'requested_batch', 'workers', 'cache', 'tf32',
                        'cudnn_benchmark', 'validation_batch_cap'}
EPOCHS = int(PLAN['epochs'])
PATIENCE = int(PLAN['patience'])
OPTIMIZER = PLAN['optimizer']
LR0 = float(PLAN['lr0'])
MOMENTUM = float(PLAN['momentum'])
SOURCE_RUN_ID = PLAN['source_run_id']

# Project choices. These are not copied from the paper.
LRF = 0.01
WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
WARMUP_BIAS_LR = 0.0
WARMUP_MOMENTUM = 0.8
COS_LR = True
OVERLAP_MASK = False
MASK_RATIO = 4
SAVE_PERIOD = 10
# Augmentation types follow the paper. Strengths are project choices.
# The Roboflow export already includes variants so no extra files are made.
AUG = dict(fliplr=0.5, flipud=0.5, degrees=10.0, translate=0.05,
           scale=0.1, hsv_h=0.01, hsv_s=0.2, hsv_v=0.2,
           mosaic=0.0, close_mosaic=0, mixup=0.0, copy_paste=0.0,
           shear=0.0, perspective=0.0, bgr=0.0)

ROOT = Path('/content/drive/MyDrive/YOLOv26')
DATASET_STORE = ROOT / 'dataset'
RUNS_ROOT = ROOT / 'runs'
RUNS_DIR = RUNS_ROOT / DATASET_NAME
ZIP_PATH = Path('/content') / f'{SOURCE_DATASET_NAME}.zip'
EXTRACT_DIR = Path('/content') / SOURCE_DATASET_NAME
DERIVED_ROOT = Path('/content') / f'{FRUIT}-two-stage-views'

print('Run:', RUN_ID, '| data profile:', FRUIT, '| stage:', STAGE)
print('Source archive:', SOURCE_DATASET_NAME, '| selected label view:', DATASET_NAME)
print('Classes:', EXPECTED_CLASSES)
print('Model:', MODEL_NAME, '| size:', IMG_SIZE, '| batch:', BATCH_SIZE)
print('Optimizer:', OPTIMIZER, '| lr0:', LR0, '| epochs:', EPOCHS, '| patience:', PATIENCE)
print('Stage 2 source Run ID:', SOURCE_RUN_ID)
print('A100 required:', REQUIRE_A100, '| RAM cache:', CACHE, '| requested workers:', WORKERS)
print('Explicit training batch:', BATCH_SIZE, '| validation cap:', EVAL_BATCH_CAP)
print('Hardware-only resume changes allowed:', ALLOW_HARDWARE_RECONFIGURE)
print('Memory recovery reductions are recorded and reused on later resumes.')
print('Tomato: Run 23 then 24. Combined apple/tomato: Run 25 then 26. New experiments require new IDs.')


Run: 26 | data profile: apple_tomato | stage: 2
Source archive: apple-tomato-source-oct01 | selected label view: apple-tomato-stage2-defects-oct01
Classes: ['bruise_discoloration', 'rot_mold_decay', 'surface_damage']
Model: yolo26l-seg.pt | size: 864 | batch: 56
Optimizer: AdamW | lr0: 0.0005 | epochs: 100 | patience: 30
Stage 2 source Run ID: 25
A100 required: True | RAM cache: ram | requested workers: 12
Explicit training batch: 56 | validation cap: 48
Hardware-only resume changes allowed: True
Memory recovery reductions are recorded and reused on later resumes.
Tomato: Run 23 then 24. Combined apple/tomato: Run 25 then 26. New experiments require new IDs.


## 1. Install and connect Drive


In [18]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        f'ultralytics=={ULTRALYTICS_VERSION}',
        'pyyaml',
        'pandas',
        'tqdm',
    ],
    check=True,
)

from google.colab import drive

drive.mount('/content/drive')

import torch
import ultralytics

assert ultralytics.__version__ == ULTRALYTICS_VERSION

print('Ultralytics:', ultralytics.__version__)

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime first.')

GPU_NAME = torch.cuda.get_device_name(DEVICE)
GPU_TOTAL_GIB = torch.cuda.get_device_properties(DEVICE).total_memory / 2**30
if REQUIRE_A100 and 'A100' not in GPU_NAME.upper():
    raise RuntimeError(f'Expected A100 but received {GPU_NAME}. Change Colab runtime to A100 first.')
print(f'GPU: {GPU_NAME}')
torch.backends.cuda.matmul.allow_tf32 = TF32
torch.backends.cudnn.allow_tf32 = TF32
torch.set_float32_matmul_precision('high' if TF32 else 'highest')
# Keep shape-specific cuDNN algorithm searches disabled for this workload.
torch.backends.cudnn.benchmark = False
print('Mixed precision:', AMP, '| TF32:', TF32, '| deterministic:', DETERMINISTIC)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics: 8.4.126
GPU: NVIDIA A100-SXM4-40GB
Mixed precision: True | TF32: True | deterministic: False


## 2. Verify and prepare the dataset

Joins the archive parts from `MyDrive/YOLOv26/dataset/`, checks the SHA-256 checksum and builds two label views.
Stage 1 keeps only the fruit class (apple, tomato, or both for the combined model). Stage 2 keeps the three defect classes as IDs 0 to 2. Images, polygon
coordinates and split assignments are not changed.

In [19]:
import collections
import hashlib
import json
import re
import shutil
import zipfile

import numpy as np
import yaml
from PIL import Image
from tqdm.auto import tqdm


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as f:
        while chunk := f.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def part_number(path):
    match = re.search(r'\.part(\d+)$', path.name)
    return int(match.group(1)) if match else None


DATASET_STORE.mkdir(parents=True, exist_ok=True)

drive_zip = DATASET_STORE / f'{SOURCE_DATASET_NAME}.zip'
part_candidates = [
    p for p in DATASET_STORE.glob(f'{SOURCE_DATASET_NAME}.zip.part*')
    if part_number(p) is not None
]
part_candidates.sort(key=part_number)

if part_candidates:
    indices = [part_number(p) for p in part_candidates]
    if indices != list(range(1, len(indices) + 1)):
        raise RuntimeError(f'Dataset parts are not contiguous from part1: {indices}')

if drive_zip.is_file() and part_candidates:
    newest_part_mtime = max(p.stat().st_mtime_ns for p in part_candidates)
    if newest_part_mtime > drive_zip.stat().st_mtime_ns:
        source_mode = 'parts'
        source_files = part_candidates
    else:
        source_mode = 'drive_zip'
        source_files = [drive_zip]
elif drive_zip.is_file():
    source_mode = 'drive_zip'
    source_files = [drive_zip]
elif part_candidates:
    source_mode = 'parts'
    source_files = part_candidates
elif ZIP_PATH.is_file():
    source_mode = 'local_zip'
    source_files = [ZIP_PATH]
else:
    raise FileNotFoundError(
        f'No dataset source found for {DATASET_NAME}. Expected either '
        f'{drive_zip} or {DATASET_NAME}.zip.part1, part2, ...'
    )

source_descriptor = [
    {
        'name': p.name,
        'size': p.stat().st_size,
        'mtime_ns': p.stat().st_mtime_ns,
    }
    for p in source_files
]
source_signature = hashlib.sha256(
    json.dumps(source_descriptor, sort_keys=True).encode()
).hexdigest()
state_path = ZIP_PATH.parent / f'.{SOURCE_DATASET_NAME}.source.json'
previous_state = json.loads(state_path.read_text()) if state_path.is_file() else {}
needs_archive_refresh = previous_state.get('source_signature') != source_signature

if source_mode == 'drive_zip':
    if needs_archive_refresh or not ZIP_PATH.is_file():
        shutil.copy2(drive_zip, ZIP_PATH)
elif source_mode == 'parts':
    if needs_archive_refresh or not ZIP_PATH.is_file():
        partial_zip = ZIP_PATH.with_suffix('.assembling')
        if partial_zip.exists():
            partial_zip.unlink()
        total_bytes = sum(p.stat().st_size for p in source_files)
        with partial_zip.open('wb') as joined, tqdm(
            total=total_bytes,
            unit='B',
            unit_scale=True,
            desc=f'Joining {DATASET_NAME}',
        ) as progress:
            for part in source_files:
                with part.open('rb') as source:
                    while chunk := source.read(8 * 1024 * 1024):
                        joined.write(chunk)
                        progress.update(len(chunk))
        partial_zip.replace(ZIP_PATH)

source_checksum = sha256_file(ZIP_PATH)
checksum = source_checksum

if EXPECTED_SHA256 and checksum.lower() != EXPECTED_SHA256.lower():
    raise RuntimeError(
        f'Dataset checksum mismatch. Expected {EXPECTED_SHA256}, got {checksum}'
    )

state_path.write_text(json.dumps({
    'source_signature': source_signature,
    'dataset_sha256': checksum,
    'source_mode': source_mode,
    'source_files': [str(p) for p in source_files],
}, indent=2))

extract_marker = EXTRACT_DIR / '.dataset_sha256'
extract_is_current = (
    EXTRACT_DIR.is_dir()
    and extract_marker.is_file()
    and extract_marker.read_text().strip() == checksum
)

if not extract_is_current:
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise RuntimeError(f'Dataset ZIP failed integrity check at {bad_member}')
        root = EXTRACT_DIR.resolve()
        for info in archive.infolist():
            target = (EXTRACT_DIR / info.filename).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f'Unsafe ZIP path: {info.filename}')
            mode = (info.external_attr >> 16) & 0o170000
            if mode == 0o120000:
                raise RuntimeError(f'Symlink entries are not allowed: {info.filename}')
        archive.extractall(EXTRACT_DIR)
    extract_marker.write_text(checksum)

yaml_candidates = list(EXTRACT_DIR.rglob('data.yaml'))
manifest_candidates = list(EXTRACT_DIR.rglob('manifest.json'))
root_candidates = sorted(
    set(p.parent for p in yaml_candidates).intersection(p.parent for p in manifest_candidates),
    key=lambda p: len(p.parts),
)

if not root_candidates:
    raise FileNotFoundError('The dataset archive must contain data.yaml and manifest.json in the same dataset root.')

DATA_DIR = root_candidates[0]
DATA_YAML = DATA_DIR / 'data.yaml'
MANIFEST_PATH = DATA_DIR / 'manifest.json'
AUDIT_PATH = DATA_DIR / 'audit.json'

manifest = json.loads(MANIFEST_PATH.read_text())
audit = json.loads(AUDIT_PATH.read_text()) if AUDIT_PATH.is_file() else {}

if not isinstance(manifest, list) or not manifest:
    raise RuntimeError('manifest.json must contain a non-empty list of image records.')

split_counts = dict(collections.Counter(str(row['split']) for row in manifest))

cfg = yaml.safe_load(DATA_YAML.read_text())
raw_names = cfg.get('names')

if isinstance(raw_names, dict):
    CLASS_NAMES = [raw_names[k] for k in sorted(raw_names, key=lambda x: int(x))]
elif isinstance(raw_names, list):
    CLASS_NAMES = list(raw_names)
else:
    raise RuntimeError('data.yaml must contain class names as a list or indexed mapping.')

if CLASS_NAMES != SOURCE_CLASSES:
    raise RuntimeError(f'Expected source classes {SOURCE_CLASSES}, received {CLASS_NAMES}.')

cfg['path'] = str(DATA_DIR)
DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False))

manifest_fruits = {
    str(row['fruit']) for row in manifest
    if row.get('fruit') not in (None, '')
}
FRUITS = sorted(manifest_fruits)
if FRUITS != sorted(WHOLE_FRUIT_CLASSES):
    raise RuntimeError(f'Expected source fruits {WHOLE_FRUIT_CLASSES}, received {FRUITS}.')
DEFECT_CLASSES = [name for name in CLASS_NAMES if name not in manifest_fruits]

validation_candidates = [name for name in ('valid', 'val', 'validation') if name in split_counts]
VALID_SPLIT_NAME = validation_candidates[0] if validation_candidates else None

hash_splits = collections.defaultdict(set)
group_splits = collections.defaultdict(set)
class_image_counts = collections.Counter()
missing_files = []

for row in tqdm(manifest, desc=f'Verifying {DATASET_NAME}'):
    split = str(row['split'])
    filename = str(row['file'])
    image = DATA_DIR / split / 'images' / filename
    label = DATA_DIR / split / 'labels' / Path(filename).with_suffix('.txt')

    if not image.is_file() or not label.is_file():
        missing_files.append((str(image), str(label)))
        continue

    with Image.open(image) as im:
        rgb = im.convert('RGB')
        digest = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()

    if row.get('pixel_hash') and digest != row['pixel_hash']:
        raise RuntimeError(f'Pixel hash mismatch: {image}')

    hash_splits[digest].add(split)

    if row.get('capture_group') not in (None, ''):
        group_splits[str(row['capture_group'])].add(split)

    classes = set()
    for line in label.read_text().splitlines():
        values = np.asarray(line.split(), dtype=float)
        if len(values) < 7 or (len(values) - 1) % 2 != 0 or not np.isfinite(values).all():
            raise RuntimeError(f'Invalid segmentation polygon: {label}')
        if values[0] != int(values[0]) or not 0 <= values[0] < len(CLASS_NAMES):
            raise RuntimeError(f'Invalid class id: {label}')
        if not ((values[1:] >= 0) & (values[1:] <= 1)).all():
            raise RuntimeError(f'Out-of-range polygon coordinate: {label}')
        classes.add(int(values[0]))

    for class_id in classes:
        class_image_counts[CLASS_NAMES[class_id]] += 1

    fruit = row.get('fruit')
    if fruit in CLASS_NAMES and CLASS_NAMES.index(fruit) not in classes:
        raise RuntimeError(f'Missing required whole-fruit mask for {fruit}: {label}')

if missing_files:
    raise FileNotFoundError(f'Missing {len(missing_files)} image/label pairs. First missing pair: {missing_files[0]}')

cross_split_hashes = {k: v for k, v in hash_splits.items() if len(v) > 1}
if cross_split_hashes:
    raise RuntimeError(f'Exact-pixel leakage detected across splits: {len(cross_split_hashes)} duplicate hashes')

cross_split_groups = {k: v for k, v in group_splits.items() if len(v) > 1}
if cross_split_groups:
    raise RuntimeError(f'Capture-group leakage detected across splits: {len(cross_split_groups)} groups')

distinct_groups = {
    split: len({
        str(row['capture_group']) for row in manifest
        if str(row['split']) == split and row.get('capture_group') not in (None, '')
    })
    for split in split_counts
}

print('Dataset:', DATASET_NAME)
print('Source:', source_mode, '| files:', len(source_files))
print('Dataset root:', DATA_DIR)
print('Split counts:', split_counts)
print('Class names:', CLASS_NAMES)
print('Fruit tags:', FRUITS or 'not available')
print('Distinct capture groups:', distinct_groups if any(distinct_groups.values()) else 'not available')
print('Images containing each class:', dict(class_image_counts))
print('Dataset SHA256:', checksum)


"""Create annotation views for whole-fruit pretraining and three-class defects.

Images and frozen split assignments are preserved. This is project scaffolding
for the students to understand, verify, and adapt under the course AI policy.
"""

import collections
import hashlib
import json
import math
import os
from pathlib import Path
import shutil

import yaml


def prepare_label_view(source, destination, keep_classes, source_sha256):
    source, destination = Path(source), Path(destination)
    keep_classes = list(keep_classes)
    raw = yaml.safe_load((source / 'data.yaml').read_text())
    names = raw['names']
    source_names = list(names) if isinstance(names, list) else [
        names[k] for k in sorted(names, key=lambda k: int(k))
    ]
    if len(set(keep_classes)) != len(keep_classes) or not set(keep_classes) <= set(source_names):
        raise ValueError('Selected classes must be distinct names from the source dataset.')
    if source.resolve() == destination.resolve():
        raise ValueError('The derived dataset must have a separate directory.')
    mapping = {source_names.index(name): j for j, name in enumerate(keep_classes)}
    manifest = json.loads((source / 'manifest.json').read_text())
    if not isinstance(manifest, list) or not manifest:
        raise ValueError('Source manifest must contain image records.')

    view_sha256 = hashlib.sha256(json.dumps({
        'source_sha256': source_sha256, 'classes': keep_classes,
        'conversion': 'keep_polygons_reindex_only_v1',
    }, sort_keys=True).encode()).hexdigest()
    marker = destination / '.label_view_sha256'
    if marker.is_file() and marker.read_text().strip() == view_sha256:
        for row in json.loads((destination / 'manifest.json').read_text()):
            label = destination / row['split'] / 'labels' / Path(row['file']).with_suffix('.txt')
            if not label.is_file() or hashlib.sha256(label.read_bytes()).hexdigest() != row.get('label_sha256'):
                raise RuntimeError(f'Cached derived label changed: {label}')
        return json.loads((destination / 'audit.json').read_text())
    if destination.exists():
        raise RuntimeError(f'Derived directory has a different identity: {destination}. Use a new dataset name.')

    temporary = destination.with_name(destination.name + '.preparing')
    if temporary.exists():
        shutil.rmtree(temporary)
    temporary.mkdir(parents=True)
    count_images, count_masks, count_empty = (collections.Counter() for _ in range(3))
    class_images, class_masks = {}, {}
    class_groups = collections.defaultdict(set)
    output_manifest = []
    seen_files = set()
    groups, hashes = collections.defaultdict(set), collections.defaultdict(set)
    try:
        for row in manifest:
            split, filename = str(row['split']), str(row['file'])
            if '/' in filename or '\\' in filename or split not in {'train', 'valid', 'val', 'test', 'test_reserve'}:
                raise ValueError('Unexpected manifest path.')
            if (split, filename) in seen_files:
                raise ValueError('Duplicate manifest record.')
            seen_files.add((split, filename))
            groups[str(row['capture_group'])].add(split)
            hashes[str(row['pixel_hash'])].add(split)
            source_image = source / split / 'images' / filename
            source_label = source / split / 'labels' / Path(filename).with_suffix('.txt')
            target_image = temporary / split / 'images' / filename
            target_label = temporary / split / 'labels' / Path(filename).with_suffix('.txt')
            if not source_image.is_file() or not source_label.is_file():
                raise FileNotFoundError(f'Missing image/label pair: {filename}')
            target_image.parent.mkdir(parents=True, exist_ok=True)
            target_label.parent.mkdir(parents=True, exist_ok=True)
            selected, present = [], set()
            for line in source_label.read_text().splitlines():
                if not line.strip():
                    continue
                parts = line.split()
                values = [float(x) for x in parts]
                if len(values) < 7 or (len(values) - 1) % 2 or not all(math.isfinite(x) for x in values):
                    raise ValueError(f'Invalid segmentation polygon: {source_label}')
                class_id = int(values[0])
                if values[0] != class_id or not 0 <= class_id < len(source_names):
                    raise ValueError(f'Invalid class ID: {source_label}')
                if not all(0 <= x <= 1 for x in values[1:]):
                    raise ValueError(f'Invalid polygon coordinates: {source_label}')
                if class_id in mapping:
                    new_id = mapping[class_id]
                    selected.append(str(new_id) + ' ' + ' '.join(parts[1:]))
                    present.add(new_id)
                    class_masks.setdefault(split, collections.Counter())[keep_classes[new_id]] += 1
            if len(keep_classes) == 1 and keep_classes[0] in {'apple', 'tomato'} and not selected:
                raise ValueError(f'No {keep_classes[0]} mask: {source_label}')
            target_label.write_text('\n'.join(selected) + ('\n' if selected else ''))
            output_manifest.append(dict(row, label_sha256=hashlib.sha256(target_label.read_bytes()).hexdigest()))
            try:
                os.link(source_image, target_image)
            except OSError:
                shutil.copy2(source_image, target_image)
            count_images[split] += 1
            count_masks[split] += len(selected)
            count_empty[split] += int(not selected)
            for class_id in present:
                class_images.setdefault(split, collections.Counter())[keep_classes[class_id]] += 1
                class_groups[split, keep_classes[class_id]].add(str(row['capture_group']))

        if any(len(v) > 1 for v in groups.values()) or any(len(v) > 1 for v in hashes.values()):
            raise ValueError('Cross-split capture groups or exact-image hashes in source dataset.')
        cfg = {'path': '.', 'train': 'train/images', 'val': raw['val'], 'test': raw['test'],
               'names': dict(enumerate(keep_classes))}
        (temporary / 'data.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False))
        (temporary / 'manifest.json').write_text(json.dumps(output_manifest, indent=2))
        audit = {
            'source_dataset': source.name, 'source_sha256': source_sha256,
            'dataset_sha256': view_sha256, 'class_names': keep_classes,
            'output_counts': dict(count_images), 'masks_by_split': dict(count_masks),
            'images_without_selected_mask_by_split': dict(count_empty),
            'images_by_class_by_split': {k: dict(v) for k, v in class_images.items()},
            'masks_by_class_by_split': {k: dict(v) for k, v in class_masks.items()},
            'distinct_capture_groups_by_class_by_split': {
                split: {name: len(class_groups[split, name]) for name in keep_classes}
                for split in count_images
            },
            'distinct_capture_groups_by_split': {
                split: len({str(row['capture_group']) for row in manifest if row['split'] == split})
                for split in count_images
            },
            'note': 'All images and frozen splits retained. Selected polygons unchanged; class IDs reindexed. Empty defect labels retained.',
        }
        (temporary / 'audit.json').write_text(json.dumps(audit, indent=2))
        (temporary / 'README.md').write_text(
            'Selected classes: ' + ', '.join(keep_classes) + '\n\n' + audit['note'] + '\n'
        )
        marker_in_temp = temporary / '.label_view_sha256'
        marker_in_temp.write_text(view_sha256)
        temporary.replace(destination)
        return audit
    except Exception:
        if temporary.exists():
            shutil.rmtree(temporary)
        raise


SOURCE_DATA_DIR = DATA_DIR
SOURCE_MANIFEST = manifest
SPLIT_SIGNATURE = hashlib.sha256(json.dumps(sorted(
    (str(r['split']), str(r['file']), str(r['capture_group']), str(r['pixel_hash']))
    for r in SOURCE_MANIFEST
), separators=(',', ':')).encode()).hexdigest()
STAGE_AUDITS = {}
for stage, class_names in STAGE_CLASSES.items():
    destination = DERIVED_ROOT / STAGE_DATASETS[stage]
    STAGE_AUDITS[stage] = prepare_label_view(
        SOURCE_DATA_DIR, destination, class_names, source_checksum
    )
    print('Label view ready:', STAGE_DATASETS[stage], '| classes:', class_names)

DATA_DIR = DERIVED_ROOT / DATASET_NAME
DATA_YAML = DATA_DIR / 'data.yaml'
MANIFEST_PATH = DATA_DIR / 'manifest.json'
AUDIT_PATH = DATA_DIR / 'audit.json'
manifest = json.loads(MANIFEST_PATH.read_text())
audit = STAGE_AUDITS[STAGE]
checksum = audit['dataset_sha256']
CLASS_NAMES = list(audit['class_names'])
if CLASS_NAMES != EXPECTED_CLASSES:
    raise RuntimeError('Derived class names do not match the selected stage.')
cfg = yaml.safe_load(DATA_YAML.read_text())
cfg['path'] = str(DATA_DIR)
DATA_YAML.write_text(yaml.safe_dump(cfg, sort_keys=False))
FRUITS = sorted({str(r['fruit']) for r in manifest if r.get('fruit')})
DEFECT_CLASSES = [c for c in CLASS_NAMES if c not in FRUITS]
print('ACTIVE DATASET:', DATASET_NAME, '| classes:', CLASS_NAMES)
print('Active split counts:', audit['output_counts'])
print('Training images per class:', audit['images_by_class_by_split'].get('train', {}))
print('Training images with no selected mask:', audit['images_without_selected_mask_by_split'].get('train', 0))
print('Split signature:', SPLIT_SIGNATURE)
print('Original image files and split assignments preserved. No image augmentation is materialized here.')


Verifying apple-tomato-stage2-defects-oct01:   0%|          | 0/4046 [00:00<?, ?it/s]

Dataset: apple-tomato-stage2-defects-oct01
Source: parts | files: 2
Dataset root: /content/apple-tomato-source-oct01
Split counts: {'train': 3423, 'test': 174, 'valid': 347, 'test_reserve': 102}
Class names: ['apple', 'tomato', 'bruise_discoloration', 'rot_mold_decay', 'surface_damage']
Fruit tags: ['apple', 'tomato']
Distinct capture groups: {'train': 1185, 'test': 170, 'valid': 342, 'test_reserve': 100}
Images containing each class: {'apple': 2467, 'bruise_discoloration': 1158, 'rot_mold_decay': 1157, 'surface_damage': 1035, 'tomato': 1579}
Dataset SHA256: 1bc76dc6eb6db1eedfe5bdf439cc8409b1eb6f8d62f76ea659a3753397e8e5d2
Label view ready: apple-tomato-stage1-whole-oct01 | classes: ['apple', 'tomato']
Label view ready: apple-tomato-stage2-defects-oct01 | classes: ['bruise_discoloration', 'rot_mold_decay', 'surface_damage']
ACTIVE DATASET: apple-tomato-stage2-defects-oct01 | classes: ['bruise_discoloration', 'rot_mold_decay', 'surface_damage']
Active split counts: {'train': 3423, 'test'

## 3. Training callbacks and run metadata

Callbacks print progress and save `experiment.json`. The saved dataset checksum and settings let an interrupted run
resume safely.

In [20]:
import hashlib
import json
import random
import time
import subprocess

import numpy as np
from ultralytics import YOLO
from ultralytics.models.yolo.segment import SegmentationTrainer


def reset_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)



def read_json(path):
    return json.loads(Path(path).read_text())


INIT_WEIGHTS = MODEL_NAME
INIT_SHA256 = None
PARENT_RUN_DIR = None
if STAGE == 2:
    if SOURCE_RUN_ID is None:
        raise ValueError('Stage 2 requires source_run_id in RUN_PLANS.')
    source_runs_dir = RUNS_ROOT / STAGE_DATASETS[1]
    matching = []
    rejected_parents = []
    for parent in source_runs_dir.glob(f'run{SOURCE_RUN_ID}_*'):
        metadata_path = parent / 'experiment.json'
        checkpoint_path = parent / 'weights' / 'best.pt'
        if not metadata_path.is_file() or not checkpoint_path.is_file():
            rejected_parents.append(f'{parent.name}: missing experiment.json or weights/best.pt')
            continue
        m = read_json(metadata_path)
        reasons = []
        if m.get('training_complete') is not True:
            reasons.append('training is not marked complete')
        if m.get('class_names') != STAGE_CLASSES[1]:
            reasons.append('whole-fruit classes differ')
        if m.get('model_name') != MODEL_NAME:
            reasons.append(f'model mismatch: saved {m.get("model_name")}, requested {MODEL_NAME}')
        if m.get('source_dataset_sha256') != source_checksum:
            reasons.append('source dataset checksum differs')
        if m.get('split_signature') != SPLIT_SIGNATURE:
            reasons.append('split assignments differ')
        if m.get('dataset_sha256') != STAGE_AUDITS[1]['dataset_sha256']:
            reasons.append('Stage 1 label view differs')
        if reasons:
            rejected_parents.append(f'{parent.name}: ' + '; '.join(reasons))
        else:
            matching.append(parent)
    if not matching:
        detail = '\n'.join(rejected_parents) or f'No Stage 1 Run {SOURCE_RUN_ID} folder found in {source_runs_dir}.'
        raise RuntimeError(
            f'Cannot initialize Stage 2 from Stage 1 Run {SOURCE_RUN_ID}.\n{detail}\n'
            'For two-stage training, both stages must use the same model preset and source partitions. '
            'A completed M run does not satisfy an L-stage preset. Train Stage 1 with L under a new '
            'Run ID, then point the L Stage 2 source_run_id to it. Loading the M checkpoint directly '
            'would construct M, even if MODEL_NAME says L.'
        )
    if len(matching) > 1:
        raise RuntimeError('More than one compatible Stage 1 folder. Use a unique source Run ID.')
    PARENT_RUN_DIR = matching[0]
    INIT_WEIGHTS = str(PARENT_RUN_DIR / 'weights' / 'best.pt')
    INIT_SHA256 = sha256_file(INIT_WEIGHTS)
    print('Stage 2 will initialize from:', INIT_WEIGHTS)
    print('New optimizer and epoch counter. Three class outputs will be rebuilt by the trainer.')


def current_training_config():
    return {
        'model_name': MODEL_NAME,
        'stage': STAGE,
        'classes': CLASS_NAMES,
        'initial_weights_sha256': INIT_SHA256,
        'imgsz': IMG_SIZE,
        'epochs': EPOCHS,
        'requested_batch': BATCH_SIZE,
        'workers': WORKERS,
        'cache': CACHE,
        'tf32': TF32,
        'cudnn_benchmark': CUDNN_BENCHMARK,
        'validation_batch_cap': EVAL_BATCH_CAP,
        'seed': SEED,
        'patience': PATIENCE,
        'amp': AMP,
        'deterministic': DETERMINISTIC,
        'optimizer': OPTIMIZER,
        'lr0': LR0,
        'lrf': LRF,
        'momentum': MOMENTUM,
        'weight_decay': WEIGHT_DECAY,
        'warmup_epochs': WARMUP_EPOCHS,
        'warmup_bias_lr': WARMUP_BIAS_LR,
        'warmup_momentum': WARMUP_MOMENTUM,
        'cos_lr': COS_LR,
        'overlap_mask': OVERLAP_MASK,
        'mask_ratio': MASK_RATIO,
        'save_period': SAVE_PERIOD,
        'augmentation': AUG,
    }


TRAINING_CONFIG = current_training_config()
TRAINING_SIGNATURE = hashlib.sha256(
    json.dumps(
        {'dataset_sha256': checksum, **TRAINING_CONFIG},
        sort_keys=True,
        separators=(',', ':'),
    ).encode()
).hexdigest()


class ProgressTrainer(SegmentationTrainer):
    def get_dataloader(self, dataset_path, batch_size=16, rank=0, mode='train'):
        # The base trainer doubles validation batch; cap it independently.
        if mode == 'val':
            batch_size = min(batch_size, EVAL_BATCH_CAP)
        return super().get_dataloader(dataset_path, batch_size, rank, mode)

    def auto_batch(self):
        # A previous run in this runtime may have enabled cuDNN benchmarking.
        torch.backends.cudnn.benchmark = False
        return super().auto_batch()

    def preprocess_batch(self, batch):
        self._last_batch_views = batch['img'].shape[0]
        return super().preprocess_batch(batch)


def check_training_setup(trainer):
    actual_batch = int(trainer.batch_size)
    auto_batch_requested = 0 < BATCH_SIZE < 1 or BATCH_SIZE == -1
    if actual_batch < 1:
        raise RuntimeError('AutoBatch did not produce a positive integer training batch.')
    expected_batch = globals().get('resume_batch') or BATCH_SIZE
    if not auto_batch_requested and actual_batch != expected_batch:
        raise RuntimeError(f'Expected actual batch {expected_batch}, received {actual_batch}.')
    torch.backends.cudnn.benchmark = CUDNN_BENCHMARK and not DETERMINISTIC
    gpu = torch.cuda.get_device_properties(DEVICE)
    if REQUIRE_A100 and 'A100' not in gpu.name.upper():
        raise RuntimeError(f'Expected A100 but received {gpu.name}.')
    if list(trainer.model.names.values()) != EXPECTED_CLASSES:
        raise RuntimeError('Trainer class names do not match the active dataset.')
    metadata_path = Path(trainer.save_dir) / 'experiment.json'
    previous = read_json(metadata_path) if metadata_path.is_file() else {}
    changes = list(previous.get('hardware_reconfigurations', []))
    if previous.get('training_signature') and previous['training_signature'] != TRAINING_SIGNATURE:
        changes.append({
            'at_start_epoch': int(getattr(trainer, 'start_epoch', 0)),
            'previous_training_config': previous.get('training_config'),
            'new_training_config': TRAINING_CONFIG,
            'previous_actual_batch': previous.get('batch'),
            'new_actual_batch': actual_batch,
        })
    report = {
        'run_id': RUN_ID,
        'dataset_name': DATASET_NAME,
        'dataset_sha256': checksum,
        'dataset_counts': split_counts,
        'class_names': CLASS_NAMES,
        'fruit_names': FRUITS,
        'capture_groups': distinct_groups,
        'model_name': MODEL_NAME,
        'stage': STAGE,
        'classes': CLASS_NAMES,
        'initial_weights_sha256': INIT_SHA256,
        'training_config': TRAINING_CONFIG,
        'training_signature': TRAINING_SIGNATURE,
        'original_training_config': previous.get('original_training_config') or previous.get('training_config') or TRAINING_CONFIG,
        'hardware_reconfigurations': changes,
        'initialization': 'stage1_checkpoint' if STAGE == 2 else 'standard_pretrained',
        'initial_weights': INIT_WEIGHTS,
        'initial_weights_sha256': INIT_SHA256,
        'source_run_id': SOURCE_RUN_ID,
        'source_dataset_sha256': source_checksum,
        'split_signature': SPLIT_SIGNATURE,
        'training_complete': False,
        'resumed_from': str(globals().get('resume_from') or ''),
        'checkpoint_selection': 'Ultralytics default box and mask mAP50-95 fitness',
        'plan': PLAN,
        'batch': actual_batch,
        'requested_batch': BATCH_SIZE,
        'imgsz': IMG_SIZE,
        'epochs': int(trainer.epochs),
        'workers': int(trainer.args.workers),
        'seed': SEED,
        'ultralytics': ultralytics.__version__,
        'gpu_name': gpu.name,
        'gpu_total_gib': gpu.total_memory / 2**30,
        'cache': CACHE,
        'tf32': TF32,
        'cudnn_benchmark': bool(torch.backends.cudnn.benchmark),
    }
    (Path(trainer.save_dir) / 'experiment.json').write_text(json.dumps(report, indent=2))
    print('Actual training batch:', actual_batch, '| loader workers:', trainer.train_loader.num_workers, flush=True)
    print('Actual validation batch:', trainer.test_loader.batch_size, '| validation workers:', trainer.test_loader.num_workers, flush=True)
    print(f'GPU: {gpu.name} | {gpu.total_memory / 2**30:.1f} GiB | cuDNN benchmark: {torch.backends.cudnn.benchmark}', flush=True)
    print('Active augmentation parameters:', AUG, flush=True)
    print('Text progress counts batches through the transform pipeline, not distinct new images.', flush=True)


def begin_epoch(trainer):
    trainer._epoch_started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats(DEVICE)
    trainer._seen_views = 0
    trainer._seen_batches = 0


def batch_progress(trainer):
    trainer._seen_batches += 1
    trainer._seen_views += trainer._last_batch_views
    if trainer._seen_batches % 50 == 0 or trainer._seen_batches == len(trainer.train_loader):
        print(
            f'  Training progress: epoch {trainer.epoch + 1}, '
            f'{trainer._seen_batches}/{len(trainer.train_loader)} batches, '
            f'{trainer._seen_views} images through training/augmentation pipeline | '
            f'{torch.cuda.memory_allocated(DEVICE) / 2**30:.1f} GiB allocated',
            flush=True,
        )


def end_epoch(trainer):
    elapsed = max(time.perf_counter() - trainer._epoch_started, 1e-6)
    gpu_report = {
        'gpu_peak_allocated_gib': torch.cuda.max_memory_allocated(DEVICE) / 2**30,
        'gpu_peak_reserved_gib': torch.cuda.max_memory_reserved(DEVICE) / 2**30,
        'training_seconds': elapsed,
        'training_images_per_second': trainer._seen_views / elapsed,
    }
    # The built-in first-epoch OOM recovery may reduce AutoBatch's estimate.
    metadata_path = Path(trainer.save_dir) / 'experiment.json'
    metadata = read_json(metadata_path)
    previous_batch = metadata.get('batch')
    metadata['batch'] = int(trainer.batch_size)
    metadata['latest_gpu_report'] = gpu_report
    temporary = metadata_path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(metadata, indent=2))
    temporary.replace(metadata_path)
    if previous_batch != metadata['batch']:
        print(f'Batch adjusted by trainer: {previous_batch} -> {metadata["batch"]}. Saved for resume.', flush=True)
    print(f'A100 progress: batch {trainer.batch_size} | {trainer._seen_views / elapsed:.1f} training images/s | '
          f'peak allocated {gpu_report["gpu_peak_allocated_gib"]:.1f} GiB | '
          f'peak reserved {gpu_report["gpu_peak_reserved_gib"]:.1f} GiB', flush=True)
    try:
        status = subprocess.run(
            ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total', '--format=csv,noheader,nounits', '-i', str(DEVICE)],
            capture_output=True, text=True, timeout=3, check=True,
        ).stdout.strip()
        print('Post-epoch GPU snapshot [utilization %, used MiB, total MiB]:', status, flush=True)
    except (OSError, subprocess.SubprocessError):
        pass  # PyTorch's memory and speed report is still available.
    with (Path(trainer.save_dir) / 'training_progress.jsonl').open('a') as f:
        f.write(json.dumps({
            'epoch': trainer.epoch + 1,
            'images_processed': trainer._seen_views,
            'actual_batch': int(trainer.batch_size),
            **gpu_report,
        }) + '\n')


def show_metrics(trainer):
    metrics = trainer.metrics or {}
    print(f'\nEPOCH {trainer.epoch + 1}/{trainer.epochs} VALIDATION', flush=True)
    for suffix, title in [('M', 'MASK'), ('B', 'BOX')]:
        values = ' | '.join(
            f'{key}: {metrics.get(f"metrics/{key}({suffix})", float("nan")):.2%}'
            for key in ['precision', 'recall', 'mAP50', 'mAP50-95']
        )
        print(title, values, flush=True)


def mark_training_complete(trainer):
    metadata_path = Path(trainer.save_dir) / 'experiment.json'
    metadata = read_json(metadata_path)
    metadata['training_complete'] = True
    metadata['completed_epochs'] = int(trainer.epoch) + 1
    metadata['finish_reason'] = 'epoch_limit' if trainer.epoch + 1 >= trainer.epochs else 'early_stopping'
    temporary = metadata_path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(metadata, indent=2))
    temporary.replace(metadata_path)
    print('Training finished:', metadata['finish_reason'], flush=True)



Stage 2 will initialize from: /content/drive/MyDrive/YOLOv26/runs/apple-tomato-stage1-whole-oct01/run25_20261001_153134/weights/best.pt
New optimizer and epoch counter. Three class outputs will be rebuilt by the trainer.


## 4. Training

Starts the selected run or resumes it from its own `last.pt` after a disconnect. A finished run is reused and not
retrained. A different dataset or setting under the same Run ID is rejected.

In [ ]:
from datetime import datetime

import pandas as pd
import torch
from ultralytics import YOLO


def values_match(saved, expected):
    if isinstance(expected, float):
        try:
            return bool(np.isclose(float(saved), expected))
        except (TypeError, ValueError):
            return False
    return saved == expected


def legacy_metadata_compatible(metadata, folder):
    if metadata.get('dataset_sha256') != checksum:
        return False
    metadata_checks = {
        'model_name': MODEL_NAME,
        'imgsz': IMG_SIZE,
        'epochs': EPOCHS,
        'requested_batch': BATCH_SIZE,
        'seed': SEED,
    }
    for key, expected in metadata_checks.items():
        if key in metadata and not values_match(metadata[key], expected):
            return False

    plan = metadata.get('plan') or {}
    if 'optimizer' in plan and not values_match(plan['optimizer'], OPTIMIZER):
        return False
    if 'lr0' in plan and not values_match(plan['lr0'], LR0):
        return False

    args_path = folder / 'args.yaml'
    if args_path.is_file():
        saved_args = yaml.safe_load(args_path.read_text()) or {}
        args_checks = {
            'imgsz': IMG_SIZE,
            'epochs': EPOCHS,
            'seed': SEED,
            'optimizer': OPTIMIZER,
            'lr0': LR0,
            'lrf': LRF,
            'momentum': MOMENTUM,
            'weight_decay': WEIGHT_DECAY,
            'warmup_epochs': WARMUP_EPOCHS,
            'warmup_bias_lr': WARMUP_BIAS_LR,
            'warmup_momentum': WARMUP_MOMENTUM,
            'cos_lr': COS_LR,
            'overlap_mask': OVERLAP_MASK,
            'mask_ratio': MASK_RATIO,
        }
        args_checks.update(AUG)
        for key, expected in args_checks.items():
            if key in saved_args and not values_match(saved_args[key], expected):
                return False

    return True


def hardware_only_compatible(metadata):
    if not ALLOW_HARDWARE_RECONFIGURE or metadata.get('dataset_sha256') != checksum:
        return False
    saved = metadata.get('training_config')
    if not isinstance(saved, dict):
        return False
    saved_core = {k: v for k, v in saved.items() if k not in HARDWARE_CONFIG_KEYS}
    current_core = {k: v for k, v in TRAINING_CONFIG.items() if k not in HARDWARE_CONFIG_KEYS}
    return saved_core == current_core


RUNS_DIR.mkdir(parents=True, exist_ok=True)
reset_seed()

same_id_runs = sorted(
    [p for p in RUNS_DIR.glob(f'run{RUN_ID}_*') if p.is_dir()],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

compatible_runs = []
incompatible_runs = []

for folder in same_id_runs:
    metadata_file = folder / 'experiment.json'
    if not metadata_file.is_file():
        continue
    metadata = json.loads(metadata_file.read_text())
    signature = metadata.get('training_signature')
    compatible = (signature == TRAINING_SIGNATURE or hardware_only_compatible(metadata)) if signature else legacy_metadata_compatible(metadata, folder)
    if compatible:
        compatible_runs.append(folder)
    else:
        incompatible_runs.append(folder)

resume_from = None
completed_run = None
saved_batch = None
resume_batch = None

if compatible_runs:
    folder = compatible_runs[0]
    last_pt = folder / 'weights' / 'last.pt'
    best_pt = folder / 'weights' / 'best.pt'
    results_csv = folder / 'results.csv'

    completed_rows = 0
    if results_csv.is_file():
        saved_history = pd.read_csv(results_csv)
        completed_rows = len(saved_history)

    metadata = json.loads((folder / 'experiment.json').read_text())
    saved_batch = int(metadata.get('batch') or metadata.get('requested_batch') or BATCH_SIZE)

    if metadata.get('training_complete') is True or completed_rows >= EPOCHS:
        completed_run = folder
        RUN_DIR = folder
        results = None
        print(f'Run {RUN_ID} is already finished: {completed_rows}/{EPOCHS} epochs (including early stopping)')
        print('Using:', RUN_DIR)
    elif not last_pt.is_file():
        if best_pt.is_file() and completed_rows > 0:
            completed_run = folder
            RUN_DIR = folder
            results = None
            print(f'Run {RUN_ID} has saved results but no resumable last.pt. Using saved run: {RUN_DIR}')
        else:
            raise RuntimeError(f'Run {RUN_ID} exists but has no checkpoint: {folder}')
    else:
        checkpoint = torch.load(last_pt, map_location='cpu', weights_only=False)
        checkpoint_epochs = int(checkpoint.get('epoch', -1)) + 1
        has_optimizer = checkpoint.get('optimizer') is not None
        del checkpoint
        completed_epochs = max(completed_rows, checkpoint_epochs)

        if not has_optimizer:
            if best_pt.is_file() and completed_rows > 0:
                completed_run = folder
                RUN_DIR = folder
                results = None
                print(
                    f'Run {RUN_ID} has {completed_rows} saved epochs but its optimizer state was stripped. '
                    f'Using the saved run without resuming.'
                )
            else:
                raise RuntimeError(f'{last_pt} has no optimizer state. Use a new RUN_ID for a fresh run.')
        else:
            resume_from = last_pt
            hardware_changed = metadata.get('training_signature') != TRAINING_SIGNATURE
            resume_batch = BATCH_SIZE if hardware_changed else saved_batch
            model = YOLO(str(resume_from))
            print(f'Resuming Run {RUN_ID} from epoch {completed_epochs}/{EPOCHS}')
            print('Checkpoint:', resume_from)
            print('Saved batch:', saved_batch, '| resumed requested batch:', resume_batch)
            if hardware_changed:
                print('Continuing checkpoint with hardware settings changed. Change is recorded in experiment.json.')

elif incompatible_runs:
    raise RuntimeError(
        f'Run ID {RUN_ID} already exists for this dataset name with a different dataset checksum or '
        f'training configuration. Use a new RUN_ID in Cell 2. Existing folder: {incompatible_runs[0]}'
    )
else:
    model = YOLO(INIT_WEIGHTS)
    run_name = f'run{RUN_ID}_' + datetime.now().strftime('%Y%m%d_%H%M%S')
    print(f'Starting fresh Run {RUN_ID}')
    print('Model:', MODEL_NAME, '| initialization:', INIT_WEIGHTS)
    print('Run folder:', run_name)

if completed_run is None:
    for event, callback in [
        ('on_train_start', check_training_setup),
        ('on_train_epoch_start', begin_epoch),
        ('on_train_batch_end', batch_progress),
        ('on_train_epoch_end', end_epoch),
        ('on_model_save', show_metrics),
        ('on_train_end', mark_training_complete),
    ]:
        model.add_callback(event, callback)

    if resume_from is not None:
        results = model.train(
            resume=True,
            trainer=ProgressTrainer,
            data=str(DATA_YAML),
            device=DEVICE,
            batch=resume_batch,
            workers=WORKERS,
            augmentations=[],
            cache=CACHE,
        )
    else:
        results = model.train(
            trainer=ProgressTrainer,
            data=str(DATA_YAML),
            project=str(RUNS_DIR),
            name=run_name,
            exist_ok=False,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            patience=PATIENCE,
            device=DEVICE,
            workers=WORKERS,
            cache=CACHE,
            amp=AMP,
            deterministic=DETERMINISTIC,
            seed=SEED,
            optimizer=OPTIMIZER,
            lr0=LR0,
            lrf=LRF,
            momentum=MOMENTUM,
            weight_decay=WEIGHT_DECAY,
            warmup_epochs=WARMUP_EPOCHS,
            warmup_bias_lr=WARMUP_BIAS_LR,
            warmup_momentum=WARMUP_MOMENTUM,
            cos_lr=COS_LR,
            resume=False,
            augmentations=[],
            overlap_mask=OVERLAP_MASK,
            mask_ratio=MASK_RATIO,
            verbose=True,
            save=True,
            save_period=SAVE_PERIOD,
            plots=True,
            **AUG,
        )

    RUN_DIR = Path(model.trainer.save_dir)
    print('Saved:', RUN_DIR)
    print('Actual training batch:', model.trainer.batch_size)


Starting fresh Run 26
Model: yolo26l-seg.pt | initialization: /content/drive/MyDrive/YOLOv26/runs/apple-tomato-stage1-whole-oct01/run25_20261001_153134/weights/best.pt
Run folder: run26_20261001_160153
New https://pypi.org/project/ultralytics/8.4.171 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, augmentations=[], auto_augment=randaugment, batch=56, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=False, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/apple_tomato-two-stage-views/apple-tomato-stage2-defects-oct01/data.yaml, degrees=10.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2

## 5. Validation metrics of the selected run

Reads `results.csv` from the run folder. Mask columns end in `(M)`.

In [21]:
import pandas as pd
import yaml

if not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')

if SAVED_RUN:
    RUN_DIR = Path(SAVED_RUN)
else:
    candidates = sorted(
        RUNS_DIR.glob(f'run{RUN_ID}_*/results.csv'),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError(f'No saved results found for Run {RUN_ID} in {RUNS_DIR}')
    RUN_DIR = candidates[0].parent

RUN_DIR = Path(RUN_DIR)

if not (RUN_DIR / 'results.csv').is_file():
    raise FileNotFoundError(f'No results.csv in {RUN_DIR}')

saved_args = yaml.safe_load((RUN_DIR / 'args.yaml').read_text()) or {}
run_metadata_path = RUN_DIR / 'experiment.json'
run_metadata = json.loads(run_metadata_path.read_text()) if run_metadata_path.is_file() else {}

saved_dataset_sha = run_metadata.get('dataset_sha256')
if saved_dataset_sha and saved_dataset_sha != checksum:
    raise RuntimeError(
        f'The selected run belongs to a different dataset checksum. '
        f'Run: {saved_dataset_sha}, active dataset: {checksum}'
    )

EVAL_BATCH_SIZE = min(EVAL_BATCH_CAP, max(1, int(run_metadata.get('batch') or saved_args.get('batch') or EVAL_BATCH_CAP)))
EVAL_IMG_SIZE = int(saved_args.get('imgsz') or run_metadata.get('imgsz') or IMG_SIZE)

history = pd.read_csv(RUN_DIR / 'results.csv')
history.columns = history.columns.str.strip()

if history.empty:
    raise RuntimeError(f'No completed epochs in {RUN_DIR}')

last = history.iloc[-1]

print('Reading:', RUN_DIR)
print('Latest completed epoch:', int(last['epoch']))

for key in ['precision', 'recall', 'mAP50', 'mAP50-95']:
    column = f'metrics/{key}(M)'
    if column in history.columns:
        print(key, f'{last[column]:.2%}')

print('Best checkpoint saved:', (RUN_DIR / 'weights' / 'best.pt').is_file())
print('Evaluation image size:', EVAL_IMG_SIZE, '| batch:', EVAL_BATCH_SIZE)


Reading: /content/drive/MyDrive/YOLOv26/runs/apple-tomato-stage2-defects-oct01/run26_20261001_160153
Latest completed epoch: 97
precision 72.62%
recall 56.19%
mAP50 57.69%
mAP50-95 35.39%
Best checkpoint saved: True
Evaluation image size: 864 | batch: 48


## 6. Confusion matrix (validation)

In [ ]:
from ultralytics import YOLO

best_checkpoint = RUN_DIR / 'weights' / 'best.pt'
if not best_checkpoint.is_file():
    raise FileNotFoundError(best_checkpoint)

best_model = YOLO(str(best_checkpoint))
eval_args = yaml.safe_load((RUN_DIR / 'args.yaml').read_text()) or {}
EVAL_MASK = {
    'overlap_mask': eval_args.get('overlap_mask', OVERLAP_MASK),
    'mask_ratio': int(eval_args.get('mask_ratio', MASK_RATIO)),
}

best_model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=EVAL_IMG_SIZE,
    batch=EVAL_BATCH_SIZE,
    device=DEVICE,
    plots=True,
    project=str(RUN_DIR),
    name='confusion_matrix_validation',
    exist_ok=True,
    **EVAL_MASK,
)


Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO26l-seg summary (fused): 207 layers, 27,906,069 parameters, 0 gradients, 139.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1420.2±503.2 MB/s, size: 47.5 KB)
val: Scanning /content/apple_tomato-two-stage-views/apple-tomato-stage2-defects-oct01/valid/labels... 347 images, 111 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 347/347 1.6Kit/s 0.2s
val: New cache created: /content/apple_tomato-two-stage-views/apple-tomato-stage2-defects-oct01/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.3it/s 6.1s
                   all        347        430      0.744      0.598      0.613      0.412      0.721      0.587        0.6      0.373
  bruise_discoloration        110        138      0.763      0.652      0.656      0.411      0.692      0.601      0.594      0.3

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e3c9e135940>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.04

## 7. Per-class mask validation

In [22]:
from ultralytics import YOLO

best_model = YOLO(str(RUN_DIR / 'weights' / 'best.pt'))
eval_args = yaml.safe_load((RUN_DIR / 'args.yaml').read_text()) or {}
EVAL_MASK = {
    'overlap_mask': eval_args.get('overlap_mask', OVERLAP_MASK),
    'mask_ratio': int(eval_args.get('mask_ratio', MASK_RATIO)),
}

print('Checkpoint:', RUN_DIR / 'weights' / 'best.pt')
print('Saved mask evaluation settings:', EVAL_MASK)

can_reuse = (
    globals().get('SAVED_RUN', '') == ''
    and
    'model' in globals()
    and 'results' in globals()
    and results is not None
    and hasattr(results, 'seg')
    and getattr(model, 'trainer', None) is not None
    and Path(model.trainer.save_dir).resolve() == RUN_DIR.resolve()
)

if can_reuse:
    scores = results
    print('Reusing final best-checkpoint validation from training.')
else:
    scores = best_model.val(
        data=str(DATA_YAML),
        split='val',
        imgsz=EVAL_IMG_SIZE,
        batch=EVAL_BATCH_SIZE,
        device=DEVICE,
        plots=False,
        verbose=True,
        project=str(RUN_DIR),
        name='best_checkpoint_validation',
        exist_ok=True,
        **EVAL_MASK,
    )


def mask_table(scores, fruit='all'):
    rows = []
    for j, class_id in enumerate(scores.seg.ap_class_index):
        rows.append({
            'fruit': fruit,
            'class_name': best_model.names[int(class_id)],
            'precision': float(scores.seg.p[j]),
            'recall': float(scores.seg.r[j]),
            'mAP50': float(scores.seg.ap50[j]),
            'mAP50_95': float(scores.seg.ap[j]),
        })
    return pd.DataFrame(rows)


def print_mask_table(table):
    for row in table.itertuples():
        print(
            f'{row.fruit:12s} {row.class_name:24s} '
            f'P {row.precision:.2%}  R {row.recall:.2%}  '
            f'mAP50 {row.mAP50:.2%}  mAP50-95 {row.mAP50_95:.2%}'
        )


per_class = mask_table(scores)
print('\nPER-CLASS MASK VALIDATION')
print_mask_table(per_class)
per_class.to_csv(RUN_DIR / 'per_class_mask_validation.csv', index=False)

(RUN_DIR / 'mask_evaluation_settings.json').write_text(json.dumps({
    'checkpoint': str(RUN_DIR / 'weights' / 'best.pt'),
    'dataset_sha256': checksum,
    'imgsz': EVAL_IMG_SIZE,
    'batch': EVAL_BATCH_SIZE,
    **EVAL_MASK,
}, indent=2))


Checkpoint: /content/drive/MyDrive/YOLOv26/runs/apple-tomato-stage2-defects-oct01/run26_20261001_160153/weights/best.pt
Saved mask evaluation settings: {'overlap_mask': False, 'mask_ratio': 4}
Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO26l-seg summary (fused): 207 layers, 27,906,069 parameters, 0 gradients, 139.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1594.5±563.4 MB/s, size: 56.4 KB)
val: Scanning /content/apple_tomato-two-stage-views/apple-tomato-stage2-defects-oct01/valid/labels... 347 images, 111 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 347/347 1.6Kit/s 0.2s
val: New cache created: /content/apple_tomato-two-stage-views/apple-tomato-stage2-defects-oct01/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.9it/s 4.2s
                   all        347        430      0.744      0.598    

292

## 8. Per-class mask validation by fruit

Fruit subsets come from the `fruit` field in `manifest.json`.

In [ ]:
from ultralytics import YOLO

if not VALID_SPLIT_NAME:
    print('Per-fruit validation skipped: no validation split was found in manifest.json.')
elif not FRUITS:
    print('Per-fruit validation skipped: manifest.json has no fruit tags.')
else:
    best_model = YOLO(str(RUN_DIR / 'weights' / 'best.pt'))
    eval_args = yaml.safe_load((RUN_DIR / 'args.yaml').read_text()) or {}
    EVAL_MASK = {
        'overlap_mask': eval_args.get('overlap_mask', OVERLAP_MASK),
        'mask_ratio': int(eval_args.get('mask_ratio', MASK_RATIO)),
    }

    fruit_tables = []

    for fruit in FRUITS:
        subset = [
            row for row in manifest
            if row.get('fruit') == fruit and str(row['split']) == VALID_SPLIT_NAME
        ]

        if not subset:
            print(f'{fruit}: no validation images, skipped')
            continue

        image_list = DATA_DIR / f'{fruit}_validation.txt'
        image_list.write_text(
            '\n'.join(
                str(DATA_DIR / str(row['split']) / 'images' / str(row['file']))
                for row in subset
            ) + '\n'
        )

        fruit_cfg = dict(cfg)
        fruit_cfg['val'] = str(image_list)
        fruit_yaml = DATA_DIR / f'{fruit}_validation.yaml'
        fruit_yaml.write_text(yaml.safe_dump(fruit_cfg, sort_keys=False))

        fruit_scores = best_model.val(
            data=str(fruit_yaml),
            split='val',
            imgsz=EVAL_IMG_SIZE,
            batch=EVAL_BATCH_SIZE,
            device=DEVICE,
            plots=False,
            verbose=True,
            project=str(RUN_DIR),
            name=f'{fruit}_mask_validation',
            exist_ok=True,
            **EVAL_MASK,
        )

        table = mask_table(fruit_scores, fruit)
        fruit_tables.append(table)

        print(f'\n{fruit.upper()}: {len(subset)} validation images')
        print_mask_table(table)

        present = set(table.class_name)
        for name in CLASS_NAMES:
            if name not in present:
                print(name, ': N/A, no evaluated ground-truth class')

    if fruit_tables:
        per_fruit = pd.concat(fruit_tables, ignore_index=True)
        output_path = RUN_DIR / 'per_class_per_fruit_mask_validation.csv'
        per_fruit.to_csv(output_path, index=False)
        print('Saved:', output_path)

        (RUN_DIR / 'per_fruit_evaluation_settings.json').write_text(json.dumps({
            'checkpoint': str(RUN_DIR / 'weights' / 'best.pt'),
            'dataset_sha256': checksum,
            'imgsz': EVAL_IMG_SIZE,
            'batch': EVAL_BATCH_SIZE,
            'fruits': FRUITS,
            **EVAL_MASK,
        }, indent=2))


Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO26l-seg summary (fused): 207 layers, 27,906,069 parameters, 0 gradients, 139.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2004.2±524.1 MB/s, size: 64.0 KB)
val: Scanning /content/apple_tomato-two-stage-views/apple-tomato-stage2-defects-oct01/valid/labels... 215 images, 59 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 215/215 1.6Kit/s 0.1s
val: New cache created: /content/apple_tomato-two-stage-views/apple-tomato-stage2-defects-oct01/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.9it/s 2.7s
                   all        215        236       0.84      0.641      0.687      0.513      0.839       0.64      0.694       0.47
  bruise_discoloration         59         67      0.892      0.742      0.787      0.573      0.874      0.727      0.782      0.52

## 9. Run history from Drive

Lists every saved run under `MyDrive/YOLOv26/runs/`. Runs on different datasets are historical and not controlled
comparisons.

In [ ]:
import re

comparison = []
result_files = sorted(RUNS_ROOT.glob('*/run*/results.csv'))

for path in result_files:
    folder = path.parent
    settings_path = folder / 'experiment.json'
    args_path = folder / 'args.yaml'

    settings = json.loads(settings_path.read_text()) if settings_path.is_file() else {}
    args = yaml.safe_load(args_path.read_text()) if args_path.is_file() else {}
    args = args or {}

    run_history = pd.read_csv(path)
    run_history.columns = run_history.columns.str.strip()

    if run_history.empty:
        continue

    last = run_history.iloc[-1]
    completed_epochs = len(run_history)
    plan = settings.get('plan') or {}
    match = re.match(r'run(\d+)', folder.name)
    fallback_run_id = int(match.group(1)) if match else -1
    run_id = settings.get('run_id', fallback_run_id)
    signature = settings.get('dataset_sha256', 'unknown')
    target_epochs = int(settings.get('epochs') or args.get('epochs') or completed_epochs)

    row = {
        'run': run_id,
        'dataset': folder.parent.name,
        'folder': folder.name,
        'model': settings.get('model_name', str(args.get('model', 'unknown'))),
        'batch': settings.get('batch', args.get('batch')),
        'completed_epochs': completed_epochs,
        'target_epochs': target_epochs,
        'complete': settings.get('training_complete', completed_epochs >= target_epochs),
        'stage': settings.get('stage', 'historical'),
        'source_run_id': settings.get('source_run_id'),
        'optimizer': plan.get('optimizer', args.get('optimizer')),
        'lr0': plan.get('lr0', args.get('lr0')),
        'extra_conv': plan.get('extra_conv', 'unknown'),
        'dataset_signature': signature,
        'same_dataset_as_active': signature == checksum,
    }

    table_path = folder / 'per_class_mask_validation.csv'

    if table_path.is_file():
        table = pd.read_csv(table_path)
        for key in ['precision', 'recall', 'mAP50', 'mAP50_95']:
            row[key] = float(table[key].mean())
        row['metric_source'] = 'saved per-class checkpoint validation'
    else:
        for key, metric in [
            ('precision', 'precision'),
            ('recall', 'recall'),
            ('mAP50', 'mAP50'),
            ('mAP50_95', 'mAP50-95'),
        ]:
            row[key] = float(last.get(f'metrics/{metric}(M)', float('nan')))
        row['metric_source'] = 'last completed epoch validation'

    comparison.append(row)

comparison = pd.DataFrame(comparison)

if comparison.empty:
    print('No saved run results found under:', RUNS_ROOT)
else:
    comparison = comparison.sort_values(['dataset', 'run', 'folder'])
    display_columns = [c for c in comparison.columns if c != 'dataset_signature']
    print(comparison[display_columns].to_string(index=False))
    print('Different dataset checksums are historical results, not controlled comparisons.')
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(RUNS_DIR / 'validation_comparison.csv', index=False)


 run                           dataset                folder                                                   model  batch  completed_epochs  target_epochs  complete      stage  source_run_id optimizer    lr0 extra_conv  same_dataset_as_active  precision   recall    mAP50  mAP50_95                         metric_source
  15            apple-one-defect-oct01 run15_20261001_000341                                          yolo26m-seg.pt   16.0                81            100     False historical            NaN       SGD 0.0100       True                   False   0.753037 0.620181 0.673356  0.422050 saved per-class checkpoint validation
  14                apple-sep30-fruit9 run14_20260930_145021                                          yolo26m-seg.pt   16.0               100            100      True historical            NaN       SGD 0.0100       True                   False   0.853374 0.653445 0.714360  0.557608 saved per-class checkpoint validation
  19          apple-stage1-whole-o

## 10. How to reproduce

1. Open the notebook in Colab with an A100 GPU runtime.
2. Set `RUN_ID` to the Stage 1 run (21 apple, 23 tomato, 25 combined). Run the configuration cell, then Sections 1 to 4.
3. Run Sections 5 to 9 to read Stage 1 validation results.
4. Restart the runtime. Set `RUN_ID` to the matching Stage 2 run (22, 24 or 26) and run the configuration cell and
   Sections 1 to 4. Stage 2 loads the Stage 1 `best.pt`.
5. If Colab disconnects keep the same Run ID and run the same cells again. Training resumes from `last.pt`.
6. Run Sections 5 to 9 for Stage 2 validation results.
7. Set `RUN_FINAL_TEST = True` only after the checkpoint is frozen and run Section 11 once.

Paper settings from Leiva et al. (2026): SGD lr0 0.001 for the fruit stage then AdamW lr0 0.0005 for the defect stage,
image size 864. YOLO26l, batch 56, patience and the augmentation strengths are project choices. The combined Stage 2
(Run 26) was capped at 100 epochs because of compute and stopped early at 97.

## 11. Unseen test set

Runs once per Stage 2 model after its checkpoint and settings were frozen: Run 22 on 79 apple test images, Run 24 on
95 tomato test images and Run 26 on all 174. Reserve images are not used. Test scores were never used to choose a run,
threshold or setting. The saved record blocks a second test on the same checkpoint.

The per-class rows printed by the first cell are labeled `apple` for every run. That label is a display bug only. The
numbers are for the run's own fruit. The second cell splits the Run 26 test by fruit using the same checkpoint and
images.

In [23]:
# Final held-out test. Enable only after model selection is frozen.
if not RUN_FINAL_TEST:
    print('Final test is disabled. Set RUN_FINAL_TEST=True in Cell 2 only when choices are frozen.')
elif STAGE != 2:
    raise RuntimeError('Final defect test requires Stage 2.')
else:
    import hashlib
    best_checkpoint = RUN_DIR / 'weights' / 'best.pt'
    if not best_checkpoint.is_file():
        raise FileNotFoundError(best_checkpoint)
    best_sha = sha256_file(best_checkpoint)
    test_record_path = RUN_DIR / 'final_test_record.json'
    if test_record_path.is_file():
        test_record = json.loads(test_record_path.read_text())
        if test_record['checkpoint_sha256'] != best_sha or test_record['dataset_sha256'] != checksum:
            raise RuntimeError('Saved test results are for a different checkpoint/dataset.')
        print('Reusing saved final test:', test_record)
    else:
        final_model = YOLO(str(best_checkpoint))
        final_scores = final_model.val(
            data=str(DATA_YAML), split='test', imgsz=EVAL_IMG_SIZE,
            batch=EVAL_BATCH_SIZE, device=DEVICE, plots=True,
            project=str(RUN_DIR), name='final_test', exist_ok=False, **EVAL_MASK,
        )
        test_table = mask_table(final_scores, 'apple')
        test_table.to_csv(RUN_DIR / 'per_class_mask_test.csv', index=False)
        test_record = {
            'checkpoint_sha256': best_sha, 'dataset_sha256': checksum,
            'split_signature': SPLIT_SIGNATURE, 'split': 'test',
            'images': split_counts.get('test', 0),
            'mask_precision': float(final_scores.seg.mp), 'mask_recall': float(final_scores.seg.mr),
            'mask_mAP50': float(final_scores.seg.map50), 'mask_mAP50_95': float(final_scores.seg.map),
        }
        test_record_path.write_text(json.dumps(test_record, indent=2))
        print('FINAL TEST MASK METRICS:', test_record)
        print_mask_table(test_table)


Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO26l-seg summary (fused): 207 layers, 27,906,069 parameters, 0 gradients, 139.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1505.0±461.4 MB/s, size: 47.6 KB)
val: Scanning /content/apple_tomato-two-stage-views/apple-tomato-stage2-defects-oct01/test/labels... 174 images, 61 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 174/174 1.6Kit/s 0.1s
val: New cache created: /content/apple_tomato-two-stage-views/apple-tomato-stage2-defects-oct01/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.1s/it 4.3s
                   all        174        192      0.637       0.49      0.457      0.307      0.621      0.511      0.454      0.278
  bruise_discoloration         43         54      0.683        0.5      0.416      0.246       0.63      0.474      0.373      0.215


In [25]:
# Per-fruit breakdown of the final test. Same frozen checkpoint and same test images.
# Adds per_fruit to final_test_record.json once and refuses to run again.
test_record_path = RUN_DIR / 'final_test_record.json'
if not test_record_path.is_file():
    raise RuntimeError('Run the final test first.')
test_record = json.loads(test_record_path.read_text())
best_checkpoint = RUN_DIR / 'weights' / 'best.pt'
if test_record['checkpoint_sha256'] != sha256_file(best_checkpoint) or test_record['dataset_sha256'] != checksum:
    raise RuntimeError('Saved test record is for a different checkpoint/dataset.')

if 'per_fruit' in test_record:
    print('Per-fruit test already done. Reusing saved record:')
    print(json.dumps(test_record['per_fruit'], indent=2))
elif len(FRUITS) < 2:
    print('Single-fruit run. The overall test is already the per-fruit result.')
else:
    final_model = YOLO(str(best_checkpoint))
    test_rows = [row for row in manifest if str(row['split']) == 'test']
    per_fruit, tables = {}, []

    for fruit in FRUITS:
        subset = [row for row in test_rows if row.get('fruit') == fruit]
        if not subset:
            print(f'{fruit}: no test images, skipped')
            continue
        image_list = DATA_DIR / f'{fruit}_test.txt'
        image_list.write_text('\n'.join(
            str(DATA_DIR / 'test' / 'images' / str(row['file'])) for row in subset
        ) + '\n')
        fruit_cfg = dict(cfg)
        fruit_cfg['test'] = str(image_list)
        fruit_yaml = DATA_DIR / f'{fruit}_test.yaml'
        fruit_yaml.write_text(yaml.safe_dump(fruit_cfg, sort_keys=False))

        scores = final_model.val(
            data=str(fruit_yaml), split='test', imgsz=EVAL_IMG_SIZE,
            batch=EVAL_BATCH_SIZE, device=DEVICE, plots=True, verbose=True,
            project=str(RUN_DIR), name=f'final_test_{fruit}', exist_ok=False, **EVAL_MASK,
        )
        per_fruit[fruit] = {
            'images': len(subset),
            'mask_precision': float(scores.seg.mp), 'mask_recall': float(scores.seg.mr),
            'mask_mAP50': float(scores.seg.map50), 'mask_mAP50_95': float(scores.seg.map),
        }
        tables.append(mask_table(scores, fruit))

    test_record['per_fruit'] = per_fruit
    test_record_path.write_text(json.dumps(test_record, indent=2))
    table = pd.concat(tables, ignore_index=True)
    table.to_csv(RUN_DIR / 'per_class_per_fruit_mask_test.csv', index=False)

    print('\nPER-FRUIT FINAL TEST MASK METRICS')
    print(json.dumps(per_fruit, indent=2))
    print()
    print_mask_table(table)


Ultralytics 8.4.126 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO26l-seg summary (fused): 207 layers, 27,906,069 parameters, 0 gradients, 139.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1973.0±311.4 MB/s, size: 59.5 KB)
val: Scanning /content/apple_tomato-two-stage-views/apple-tomato-stage2-defects-oct01/test/labels... 79 images, 17 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 79/79 1.2Kit/s 0.1s
val: New cache created: /content/apple_tomato-two-stage-views/apple-tomato-stage2-defects-oct01/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.1s/it 2.2s
                   all         79         89      0.753      0.569      0.563      0.422      0.769      0.591      0.586      0.401
  bruise_discoloration         20         23      0.702      0.513      0.415      0.304      0.703      0.516      0.411      0.278
   

## 12. Run comparison

Every recorded run from the first pilot to the final model. Validation mask metrics. Pilots P1 to P6 had no Run ID. Runs 4, 8, 10, 12, 13 and 16 to 18 were stopped before producing metrics and are left out.

| Run | Date | Data | Classes | Train | Model | Epochs | P | R | mAP50 | mAP50-95 | Note |
|---|---|---|---|---:|---|---|---:|---:|---:|---:|---|
| P1 five-class pilot | 2026-09-13 | Fruit-Segmentation-v5 | apple+bruise_discoloration+cut_crack+rot_mold_decay+surface_spot_scar | 252 | yolo26n-seg | 50/50 | 0.640 | 0.458 | 0.474 | 0.310 | first five-class pilot |
| P2 combined-defect pilot | 2026-09-13 | Fruit-Segmentation-v5 combined | apple+defect | 252 | yolo26n-seg | 96/100 | 0.827 | 0.750 | 0.751 | 0.523 | all defects merged into one class |
| P3 export5 pilot | 2026-09-17 | apple-pilot-export5 | apple+bruise_discoloration+cut_crack+rot_mold_decay+surface_spot_scar | 767 | yolo26n-seg | 50/50 | 0.728 | 0.503 | 0.498 | 0.329 | more training images same five classes |
| P4 mask-and-defect pilot | 2026-09-17 | apple-mask-and-defect-pilot | apple_mask+defect | 767 | yolo26n-seg | 50/50 | 0.745 | 0.667 | 0.700 | 0.513 | two classes on export5 images |
| P5 three-fruit pilot | 2026-09-20 | three-fruit-first-pilot | fruit x3+bruise_discoloration+rot_mold_decay+surface_damage | 3360 | yolo26s-seg | 50/50 | 0.757 | 0.684 | 0.683 | 0.526 | mango dropped afterwards |
| P6 online-aug pilot | 2026-09-24 | apple-tomato-online-aug-v1 | apple+tomato+bruise_discoloration+rot_mold_decay+surface_damage | 976 | yolo26s-seg | 100/100 | 0.785 | 0.674 | 0.687 | 0.506 | progress report baseline |
| Run 1 | 2026-09-26 | apple-tomato-sep26 | apple+tomato+3 defects | 1160 | yolo26s-seg | 50/50 | 0.721 | 0.630 | 0.626 | 0.443 |  |
| Run 2 | 2026-09-26 | apple-tomato-sep26 | apple+tomato+3 defects | 1160 | yolo26s-seg | 50/50 | 0.770 | 0.704 | 0.739 | 0.619 | lower learning rate |
| Run 2b | 2026-09-26 | apple-tomato-sep26 | apple+tomato+3 defects | 1160 | yolo26s-seg | 100/100 | 0.801 | 0.708 | 0.742 | 0.618 | AdamW 100 epochs |
| Run 3 | 2026-09-26 | apple-tomato-sep26 | apple+tomato+3 defects | 1160 | yolo26s-seg + extra conv | 50/50 | 0.675 | 0.624 | 0.611 | 0.426 | extra conv layer did not help |
| Run 5 | 2026-09-27 | apple-tomato-sep27-members | apple+tomato+3 defects | 1285 | yolo26s-seg | 100/100 | 0.817 | 0.697 | 0.747 | 0.622 | member annotations added |
| Run 6 | 2026-09-27 | apple-tomato-sep27-members 70/20/10 | apple+tomato+3 defects | 1215 | yolo26m-seg | 100/100 | 0.828 | 0.717 | 0.750 | 0.622 | larger model and 70/20/10 split |
| Run 7 | 2026-09-28 | apple-tomato-sep27-members 70/20/10 | apple+tomato+3 defects | 1215 | yolo26l-seg | 35/100 | 0.686 | 0.643 | 0.647 | 0.540 | stopped early (memory limit) |
| Run 11 | 2026-09-29 | apple-tomato-sep29-refresh | apple+tomato+3 defects | 3423 | yolo26m-seg | 100/100 | 0.809 | 0.690 | 0.715 | 0.598 | best combined apple and tomato run |
| Run 14 | 2026-09-30 | apple-sep30-fruit9 | apple+3 defects | 2110 | yolo26m-seg | 100/100 | 0.853 | 0.653 | 0.714 | 0.558 | single-stage apple baseline |
| Run 15 | 2026-10-01 | apple-one-defect-oct01 | defect | 2110 | yolo26m-seg | 81/100 | 0.753 | 0.620 | 0.673 | 0.422 | one merged defect class (unfinished) |
| Run 19 (Stage 1) | 2026-10-01 | apple-stage1-whole-oct01 | apple | 2110 | yolo26m-seg | 21/100 | 0.991 | 0.981 | 0.985 | 0.985 | two-stage method Stage 1 (26m) |
| Run 20 (Stage 2) | 2026-10-01 | apple-stage2-defects-oct01 | 3 defects | 2110 | Run 19 best.pt (26m) | 91/300 | 0.812 | 0.632 | 0.671 | 0.450 | two-stage method Stage 2 (26m) |
| Run 21 (Stage 1) | 2026-10-01 | apple-stage1-whole-oct01 | apple | 2110 | yolo26l-seg | 34/100 | 1.000 | 0.986 | 0.985 | 0.985 | final Stage 1 (26l) |
| Run 22 (Stage 2) | 2026-10-01 | apple-stage2-defects-oct01 | 3 defects | 2110 | Run 21 best.pt (26l) | 124/300 | 0.809 | 0.680 | 0.688 | 0.457 | final Stage 2 (26l). Test mAP50 0.613 |
| Run 23 (Stage 1) | 2026-10-01 | tomato-stage1-whole-oct01 | tomato | 1313 | yolo26l-seg | 14/100 | 0.935 | 0.972 | 0.966 | 0.959 | tomato Stage 1 (26l) |
| Run 24 (Stage 2) | 2026-10-01 | tomato-stage2-defects-oct01 | 3 defects | 1313 | Run 23 best.pt (26l) | 190/300 | 0.709 | 0.473 | 0.494 | 0.295 | final tomato Stage 2 (26l). Test mAP50 0.393 |
| Run 25 (Stage 1) | 2026-10-01 | apple-tomato-stage1-whole-oct01 | apple+tomato | 3423 | yolo26l-seg | 14/100 |  |  |  |  | combined Stage 1. Apple mAP50 0.985 tomato 0.949 |
| Run 26 (Stage 2) | 2026-10-01 | apple-tomato-stage2-defects-oct01 | 3 defects | 3423 | Run 25 best.pt (26l) | 97/100 | 0.721 | 0.587 | 0.600 | 0.374 | combined Stage 2. Test mAP50 apple 0.586 tomato 0.352 |

![Run history](../reports/figures/run_history_map50.png)

Runs 14, 20 and 22 use the same images and splits. Averaged over the three defect classes the single-stage Run 14 scored 0.624 mask mAP50, two-stage YOLO26m Run 20 scored 0.671 and two-stage YOLO26l Run 22 scored 0.688. These runs also differ in model size, optimizer and image size so the gain is not proof that two stages alone helped.

![Defect comparison](../reports/figures/defect_comparison.png)

## 13. Web deployment

The Streamlit app is in the repository root as `streamlit_app.py`. It runs locally.

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
streamlit run streamlit_app.py
```

Place the checkpoints in `models/`: `apple_stage1.pt` (Run 21 best.pt) and `apple_stage2.pt` (Run 22 best.pt).
Tomato uses `tomato_stage1.pt` (Run 23) and `tomato_stage2.pt` (Run 24). The combined model uses
`apple_tomato_stage1.pt` (Run 25) and `apple_tomato_stage2.pt` (Run 26).

Features: fruit selector, upload or sample image, colored masks with legend, defect share of the image and of the
Stage 1 fruit mask, confidence slider, class toggles, inference time, PNG download and clear messages for unreadable
files or images with no defect.

![App screenshot](../reports/figures/app_screenshot.png)

## 14. Results and discussion

**Apple Stage 1 (Run 21, validation).** Mask precision 1.000, recall 0.986,
mAP50 0.985, mAP50-95 0.985.

**Apple Stage 2 (Run 22).**

| Metric | Validation | Test (79 images) |
|---|---:|---:|
| Mask precision | 0.809 | 0.718 |
| Mask recall | 0.680 | 0.597 |
| Mask mAP50 | 0.688 | 0.613 |
| Mask mAP50-95 | 0.457 | 0.411 |

| Class | Val P | Val R | Val mAP50 | Test mAP50 |
|---|---:|---:|---:|---:|
| Bruise | 0.847 | 0.744 | 0.777 | 0.594 |
| Rot | 0.931 | 0.713 | 0.761 | 0.732 |
| Surface | 0.648 | 0.584 | 0.527 | 0.511 |

**IoU.** Mean per-image pixel IoU between predicted and labeled defect masks (confidence 0.25) over the 62 test images
with a labeled defect: 0.583.

**Learning curves.** Training loss falls smoothly and validation mAP levels off after about 30 epochs, so early
stopping ended the runs. The tomato run (Run 24) shows overfitting: its validation segmentation loss was lowest near
epoch 26 and rose afterwards while training loss kept falling. The checkpoint was chosen on validation fitness.

![Learning curves](../reports/figures/learning_curves.png)

**Validation vs test.**

![Validation vs test](../reports/figures/val_vs_test.png)

**Damaged share of the fruit.** The app reports defect area as a share of the Stage 1 fruit mask. Predicted and
labeled shares agree with r = 0.91 and a mean error of 5.2 points on apple test images, and r = 0.82 and 3.7 points
on tomato. Most large errors are defects missed entirely.

![Coverage agreement](../reports/figures/coverage_agreement.png)

**Two stages in action.**

![Two-stage example](../reports/figures/two_stage_example.png)

![Good predictions](../reports/figures/good_predictions.png)

![Failure cases](../reports/figures/failure_cases.png)

**Strengths.** The whole fruit is segmented almost perfectly. Bruise and rot are rarely confused with each other.

**Weaknesses.** Most errors are missed defects. Surface damage is the hardest class because the marks are small and
low contrast. Most false positives are surface damage on marks that were not labeled.

**Edge cases.** Defects at the fruit edge, glare, natural speckles, busy backgrounds and photos unlike the training
sources. The model has not been tested on conveyor or packing-line images.

**Tomato (Runs 23 and 24).** Same two-stage recipe on 1,313 training files from 459 capture groups.

| Metric | Validation | Test (95 images) |
|---|---:|---:|
| Stage 1 whole tomato mAP50 | 0.966 | - |
| Stage 2 mask precision | 0.709 | 0.732 |
| Stage 2 mask recall | 0.473 | 0.375 |
| Stage 2 mask mAP50 | 0.494 | 0.393 |
| Stage 2 mask mAP50-95 | 0.295 | 0.194 |

Per class test mask mAP50: bruise 0.388, rot
0.477, surface damage
0.314. Precision is reasonable but recall is low: over half of each
tomato defect class is missed as background on test.

![Tomato test confusion matrix](../runs/run24_stage2_tomato_yolo26l/final_test/confusion_matrix_normalized.png)

Mean per-image pixel IoU over 51 tomato test images with a defect: 0.41. Large rot regions are found well. Faint
bruises that are only a slight color change are missed.

![Tomato examples](../reports/figures/tomato_examples.png)

**One model for both fruits (Runs 25 and 26).** The apple and tomato splits were merged without reshuffling. Stage 1
learns two classes (apple, tomato) and Stage 2 keeps the three shared defect names, so the fruit comes from Stage 1.
Stage 1 reached 0.985 mask mAP50 for apple and 0.949 for tomato on
validation. Test mask mAP50 by fruit:

| Class | Apple separate (Run 22) | Apple combined (Run 26) | Tomato separate (Run 24) | Tomato combined (Run 26) |
|---|---:|---:|---:|---:|
| Bruise | 0.594 | 0.411 | 0.388 | 0.360 |
| Rot | 0.732 | 0.849 | 0.477 | 0.442 |
| Surface | 0.511 | 0.497 | 0.314 | 0.253 |
| All | 0.613 | 0.586 | 0.393 | 0.352 |

The combined model was level with the separate models on validation and slightly lower on test for both fruits.
Each class has only 18 to 28 test images so these gaps are within the noise of one split. Combining did not improve
accuracy but one model can serve both fruits at a small cost.

![Separate vs combined](../reports/figures/separate_vs_combined.png)

## 15. IEEE paper overview

The paper `paper/CIPHER_AI2_IEEE_Paper.docx` follows the IEEE conference format:

- **Abstract** with the task, data, method and test result
- **Introduction** with the problem and contributions
- **Related literature** on grading data, defect segmentation, SAM-assisted labels and robustness
- **Dataset and annotation** with sources, classes and leakage control
- **Methodology** with the two-stage training and metrics
- **Results and discussion** with run history, validation, unseen test, tomato, the combined model and errors
- **Conclusion and future work**
- **References** in IEEE style

## 16. Insights and conclusions

- Clear class definitions mattered more than model size early on. Merging rare and confusing classes raised scores.
- Overall mAP that includes the whole-fruit class hides weak defect results. Defect classes must be reported alone.
- Splitting by capture group was needed because the Roboflow export repeats each photo up to three times.
- The two-stage YOLO26l model gave the best defect score on the same split. The gain is not isolated from model size.
- Small surface damage remains the main open problem. More examples and finer masks are the next step.
- Tomato is the weakest fruit. Its model is precise but misses most defects, which points to too few examples.
- One combined model for both fruits came close to two separate models but did not beat them on test.

## 17. References

Agarla, M., Napoletano, P., & Schettini, R. (2023). Quasi real-time apple defect segmentation using deep learning. *Sensors, 23*(18), 7893. https://doi.org/10.3390/s23187893

He, K., Gkioxari, G., Dollár, P., & Girshick, R. (2017). Mask R-CNN. *Proceedings of the IEEE International Conference on Computer Vision*, 2961–2969.

Kirillov, A., et al. (2023). Segment anything. *Proceedings of the IEEE/CVF International Conference on Computer Vision*, 4015–4026.

Knott, M., Odion, D., Sontakke, S., Karwa, A., & Defraeye, T. (2025). Weakly supervised panoptic segmentation for defect-based grading of fresh produce. *CVPR Workshops*.

Leiva, F., Rebeaud, S. G., & Christen, D. (2026). Real-time identification and quantification of apple scab on fruit in preharvest and postharvest conditions using YOLO11: A deep learning approach. *Plant Methods, 22*, 29. https://doi.org/10.1186/s13007-026-01508-7

Mojumdar, M. U., et al. (2025). AFruitDB: A comprehensive dataset of six commonly used Asian fruits for advanced grading and biodiversity insights. *Data in Brief, 59*, 111380. https://doi.org/10.1016/j.dib.2025.111380

Nesteruk, S. (n.d.). *Apple rotting segmentation problem in the wild* [Data set]. Kaggle. CC BY-NC-SA 4.0. https://www.kaggle.com/datasets/sergeynesteruk/apple-rotting-segmentation-problem-in-the-wild

Nesteruk, S., et al. (2022). XtremeAugment: Getting more from your data through combination of image collection and image augmentation. *IEEE Access, 10*, 24010–24028. https://doi.org/10.1109/ACCESS.2022.3154709

Tan, C., Li, C., Perkins-Veazie, P., Oh, H., Xu, R., & Iorizzo, M. (2025). High throughput assessment of blueberry fruit internal bruising using deep learning models. *Frontiers in Plant Science, 16*, 1575038. https://doi.org/10.3389/fpls.2025.1575038

Ultralytics. (2026). *YOLO26*. https://docs.ultralytics.com/models/yolo26/

### AI tool disclosure

AI tools used: Claude (Anthropic) and OpenAI Codex. They were used for planning, explaining concepts, drafting text
and scaffolding code. The team reviewed, tested and edited every output and can explain each part. The complete
prompt and response transcripts are included in `ai_usage/`.

# ⚠️ Notice

## This material is the intellectual property of <font color="orange">Dr. Lysa V. Comia.</font>

Redistribution, reproduction, or use beyond personal reference is strictly prohibited without the prior written consent of the author.

## 📌 Notice on Use, Redistribution, and Ethical Compliance

Redistribution, reproduction, or use of this material beyond personal reference is **strictly prohibited** without the prior written consent of the author. Unauthorized copying, modification, or dissemination, whether for commercial, academic, or institutional purposes, violates intellectual property rights and may result in legal or disciplinary action.

### 🔒 AI Governance and Ethics Considerations

This work must **not** be used in ways that:

- ❌ **Compromise data privacy** or violate data protection regulations (e.g., GDPR, Philippine Data Privacy Act).
- ❌ **Perpetuate bias or discrimination** by misusing algorithms, datasets, or results.
- ❌ **Enable harmful applications**, including surveillance, profiling, or uses that undermine human rights.
- ❌ **Misrepresent authorship or credit**, such as plagiarism or omission of proper citations.

### ✅ Responsible Use Principles

Users are expected to follow responsible research and innovation practices, ensuring that any derivative work is:

- **Transparent**: Clear acknowledgment of sources and methodology.
- **Accountable**: Proper attribution and disclosure of limitations.
- **Beneficial to society**: Applications that align with ethical standards and do not cause harm.

📢 For **any intended use** (academic, research, or practical), prior written approval must be obtained from the author to ensure compliance with both **legal requirements** and **ethical AI practices**.